# 1 — Compute
Full analysis **without any figure**: frame clustering (Leiden + UMAP), state sequences,
SeqVec strategies (explorer, no window), optimal regrouping, cluster statistics, session save.

Run all cells from top to bottom, then set `DO_SAVE = True` (and a `SAVE_NAME`) in the save cell
and rerun it. The other notebooks (`2_model`, `3_results`, `4_interactive`) start from that session.

In [3]:
# ── Repository folders (RELATIVE paths: this notebook is in Code/) ──
from pathlib import Path
REPO_DIR    = Path.cwd().parent if Path.cwd().name == "Code" else Path.cwd()
DATA_DIR    = REPO_DIR / "Data"       # feature CSVs, tracking, per-individual Excel
RESULTS_DIR = REPO_DIR / "Results"    # saved sessions (e.g. reference_best) + outputs
VIDEO_ROOT  = REPO_DIR / "Videos"     # videos NOT included in the repository: put them here
_LOCAL_VIDEOS = Path("E:/01_Travail/CREC/First_Capture/VIDEO/capture")   # local copy (author's PC)
if not VIDEO_ROOT.is_dir() and _LOCAL_VIDEOS.is_dir():   # no Videos/ -> local videos
    VIDEO_ROOT = _LOCAL_VIDEOS
VIDEO_DIR_CUT  = VIDEO_ROOT / "videos_catch"       # SHORT videos (videoN_cut.mp4): cuttlefish sessions
VIDEO_DIR_FULL = VIDEO_ROOT / "01_VIDEO_Cropped"   # LONG videos (videoN.mp4): shrimp sessions / DATASET "full"

import json
import webbrowser
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import kneighbors_graph
import igraph as ig
import leidenalg
import umap
import pickle

%matplotlib inline
#%matplotlib qt

# ══ SUBJECT: whose behaviour is analysed ═════════════════════
SUBJECT = "cuttle"    # "cuttle" = cuttlefish (files chosen by DATASET below)
                      # "shrimp" = shrimp (add_features_shrimp.py): 30 s before the start of the
                      #            cuttlefish activity -> strike; column "phase" = "pre" / "hunt",
                      #            the change is marked on the figures (dashed cyan line)
SMOOTH_FILES_SHRIMP = {
    1:  DATA_DIR / "shrimp_features_smooth_1.csv",
}
FEATURES_SHRIMP = [
    "shrimp_speed_log",       # speed of the shrimp (log)
    "shrimp_burst_rate",      # acceleration peaks per second (sudden starts / tail-flips)
    "shrimp_forward_log",     # swimming direction: + forwards / - backwards / 0 still
    "shrimp_turn_log",        # angular speed of the body: turns on itself (log)
    "shrimp_view_cuttle",     # cuttlefish seen by the shrimp: 0° = in front of its head, 180° = behind
    "distance_log",           # distance to the cuttlefish (log)
]
FEATURE_WEIGHTS_SHRIMP = {f: 1.0 for f in FEATURES_SHRIMP}

# ══ DATASET CHOICE ═══════════════════════════════════════════
DATASET = "cut_new"   # "cut_old" = old cut file  + old features (5)
                      # "cut_new" = new cut file  + new features (9)
                      #             (Frame_start -> strike, ~95 k frames)
                      # "full"    = full file     + new features (9)
                      #             (Start -> strike, ~1.2 M frames)

# ── CSV files
SMOOTH_FILES_CUT_OLD = {     # add_features_cut_2.py (old computation)
    1:  DATA_DIR / "cut_cuttlefish_with_features_smooth_1.csv",
   # 5:  DATA_DIR / "cut_cuttlefish_with_features_smooth_5.csv",
    #10: DATA_DIR / "cut_cuttlefish_with_features_smooth_10.csv",
    #20: DATA_DIR / "cut_cuttlefish_with_features_smooth_20.csv",
    #50: DATA_DIR / "cut_cuttlefish_with_features_smooth_50.csv",
}
SMOOTH_FILES_CUT_NEW = {     # add_features_full.py, SOURCE = "cut"
    1:  DATA_DIR / "cut_cuttlefish_with_new_features_smooth_1.csv",
   # 5:  DATA_DIR / "cut_cuttlefish_with_new_features_smooth_5.csv",
    #10: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_10.csv",
    #20: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_20.csv",
    #50: DATA_DIR / "cut_cuttlefish_with_new_features_smooth_50.csv",
}
SMOOTH_FILES_FULL = {        # add_features_full.py, SOURCE = "full"
    1:  DATA_DIR / "full_cuttlefish_with_features_smooth_1.csv",
}

# ── Features
FEATURES_OLD = ["distance_to_shrimp", "approach_angle", "speed_cuttle_smooth",
                "orientation_variation", "jerk_cuttle"]

# FEATURES_OLD = [
#     "distance_to_shrimp", "approach_angle", "orientation_variation",
#     "speed_cuttle_smooth", "accel_cuttle" ,"jerk_cuttle", "head_turn", "eye_convergence"
# ]

# New variables (add_features_full.py) — all except distance_change
FEATURES_NEW = [
    # 6 variables, each = one behavioural axis
    "distance_to_shrimp",        # near or far from the prey (log)
    "attack_side",               # where the cuttlefish is around the prey: 0° = in front of its head, 180° = behind
    "bearing_to_prey",           # is the cuttlefish aiming at the prey? 0° = straight ahead
    "cuttle_approach_speed",     # cuttlefish moves towards the prey (+) / backs off (-) / still (0)
    "orbit_speed",               # cuttlefish turns around the prey (°/s) -> circling
    "who_moves",                 # who moves: + = cuttlefish, - = shrimp, 0 = both / neither
    # Available in the file but removed from the clustering:
    # "speed_cuttle_log",        # redundant: approach + orbit already decompose the movement
    # "orientation_variation",   # correlated with speed (r=0.79), rotation covered by orbit_speed
    # "approach_angle",          # relative body orientation, replaced by attack_side
    # "approach_angle_variation",# half driven by the shrimp, replaced by orbit_speed
    # "speed_shrimp_log",        # prey behaviour (context, to describe afterwards)
    # "shrimp_retreat_speed",    # same
    # "distance_change",         # mixes cuttlefish + shrimp (r=0.87 with shrimp_retreat_speed)
]

# ── Feature weights (1.0 = no weighting) ────────────────────
#   Applied AFTER the StandardScaler: a weight W gives the feature W times
#   more importance in the distance between frames.
#   A feature missing from the dictionary has a weight of 1.0.
FEATURE_WEIGHTS_OLD = {      # for "cut_old" (current weights taken from cell 1)
    "orientation_variation":          8.0,
    "distance_to_shrimp":             1.0,
    "approach_angle":                 1.0,
    "speed_cuttle_smooth":            1.0,
    "jerk_cuttle":                    1.0
}

FEATURE_WEIGHTS_NEW = {      # for "cut_new" and "full"
    "distance_to_shrimp":        1.0,
    "attack_side":               3.0,
    "bearing_to_prey":           1.0,
    "cuttle_approach_speed":     2.0,   # movement: higher weight so that Leiden
    "orbit_speed":               5.0,   #   splits states by MOVEMENT rather than
    "who_moves":                 4.0,   #   by position alone
}

# ── Excluded individuals (dubious data) ──────────────────────
EXCLUDE_ANIMALS = [
    152,    # anomalous cut file (4314 frames instead of ~251) -> creates its own state
    133,    # missing from the Excel (no reliable strike frame) -> creates 2 states on its own
]

# ── Orbit threshold ───────────────────────────────────────────
#   orbit_speed is recomputed at loading (cell 2) from orbit_speed_deg_s:
#   below ORBIT_MIN_DEG_S (°/s) = slight curve, not circling -> 0, then log.
#   Without a threshold, log1p makes 3-6 °/s almost as strong as 15-50 °/s.
#   Can be changed here without rerunning add_features_full.py (0 = no threshold).
ORBIT_MIN_DEG_S = 0.0   # tested: 8 degraded the groups (single orbit state); 0 or 3 advised

# ── Leiden parameters
LEIDEN_K    = 300
LEIDEN_RES  = 1.0
LEIDEN_K2   = 3
LEIDEN_RES2 = 5.0

# ── Settings depending on DATASET ───────────────────────────
#   FRAME_STEP: Leiden + UMAP on 1 frame out of FRAME_STEP (the others: by neighbours)
FRAME_STEP = 1
VIDEO_DIR  = VIDEO_DIR_CUT   # videoN_cut.mp4

if DATASET == "cut_old":
    SMOOTH_FILES = SMOOTH_FILES_CUT_OLD
    FEATURES     = list(FEATURES_OLD)
    OUT_DIR      = RESULTS_DIR / "analysis_results"
elif DATASET == "cut_new":
    SMOOTH_FILES = SMOOTH_FILES_CUT_NEW
    FEATURES     = list(FEATURES_NEW)
    OUT_DIR      = RESULTS_DIR / "analysis_results_cut_new"
    LEIDEN_K     = 300   # K=300 / res=0.5: more homogeneous clusters than with K=1000
    LEIDEN_RES   = 1.0   #   res 1.0: states that separate the orbit better (user choice)
elif DATASET == "full":
    SMOOTH_FILES = SMOOTH_FILES_FULL
    FEATURES     = list(FEATURES_NEW)
    OUT_DIR      = RESULTS_DIR / "analysis_results_full"
    VIDEO_DIR    = VIDEO_DIR_FULL   # videoN.mp4
    FRAME_STEP   = 3     # ~400 k frames for Leiden instead of 1.2 M
    LEIDEN_K     = 200   # 1000 neighbours × 400 k frames does not fit in memory
    LEIDEN_RES   = 0.3   # K=200 / res=0.3 -> ~17 clusters (comparable to the cut)
                         # if MemoryError: LEIDEN_K = 100 and LEIDEN_RES = 0.1
else:
    raise ValueError(f"DATASET = {DATASET!r} unknown — 'cut_old', 'cut_new' or 'full'")

USE_FULL = DATASET == "full"          # full file?
USE_NEW  = DATASET != "cut_old"       # new features?
FEATURE_WEIGHTS = dict(FEATURE_WEIGHTS_NEW if USE_NEW else FEATURE_WEIGHTS_OLD)
SPEED_COL = "speed_cuttle_mm_s"       # speed used by the SeqVec "speed" and "path" families
if SUBJECT == "shrimp":
    DATASET         = "shrimp"
    SMOOTH_FILES    = SMOOTH_FILES_SHRIMP
    FEATURES        = list(FEATURES_SHRIMP)
    FEATURE_WEIGHTS = dict(FEATURE_WEIGHTS_SHRIMP)
    OUT_DIR         = RESULTS_DIR / "analysis_results_shrimp"
    SPEED_COL       = "shrimp_speed_mm_s"
    LEIDEN_K, LEIDEN_RES = 300, 1.0
    FRAME_STEP      = 3       # ~365 k frames (30 s pre + hunt): Leiden + UMAP on 1 frame / 3
    USE_FULL        = True    # frames numbered as in the full videos (01_VIDEO_Cropped)
    VIDEO_DIR       = VIDEO_DIR_FULL
elif SUBJECT != "cuttle":
    raise ValueError(f"SUBJECT = {SUBJECT!r} unknown — 'cuttle' or 'shrimp'")
_unknown = [f for f in FEATURE_WEIGHTS if f not in FEATURES]
if _unknown:
    print(f"[WARN] weights defined for unused features (ignored): {_unknown}")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Palettes
PALETTE = [
    "#FF4FB8","#C97E48","#8C6BB1","#9C9C9C","#F5E21E",
    "#3A8891","#E45756","#4CAF50","#1F77B4","#FF9F1C",
    "#A05195","#2CB1BC","#7F7F7F","#BCBD22","#17BECF",
    "#D62728","#9467BD","#8C564B","#E377C2","#AEC7E8",
]
SUPER_PAL = [
    "#FF4444","#44AAFF","#44FF88","#FFD700","#FF44FF",
    "#FF8844","#44FFFF","#FF44AA","#88FF44","#AA44FF",
]
# ── Semantic aliases ─────────────────────────────────
CLUSTER_PALETTE  = PALETTE    # behavioural states C0, C1, ...
STRATEGY_PALETTE = SUPER_PAL  # strategies A, B, C, ... (and Ward groups)

# Fixed colormaps — single source of truth
CMAP_CLUSTER = mpl.colors.ListedColormap(PALETTE)
NORM_CLUSTER = mpl.colors.BoundaryNorm(range(len(PALETTE) + 1), ncolors=len(PALETTE))

# ── Shrimp sessions: change of phase "pre" -> "hunt" (start of the cuttlefish activity) ──
def phase_change(df, animal_id=None):
    """Position of the pre -> hunt change (column 'phase', shrimp data only; None otherwise).
    animal_id given -> index of the first 'hunt' frame in the frame-sorted sequence of this individual;
    otherwise -> {animal_id: position 0..1 in normalised time}."""
    if df is None or "phase" not in getattr(df, "columns", []):
        return None
    if animal_id is not None:
        ph = df.loc[df["animal_id"] == animal_id].sort_values("frame")["phase"].to_numpy()
        return int(np.argmax(ph == "hunt")) if (ph == "pre").any() and (ph == "hunt").any() else None
    pre = (df["phase"] == "pre").groupby(df["animal_id"]).mean()
    return {int(a): float(v) for a, v in pre.items()}

def phase_median(df, members):
    """Median normalised position of the phase change over a list of individuals (None if not available)."""
    ph = phase_change(df)
    v = [ph[int(m)] for m in members if ph and str(m).lstrip("-").isdigit() and int(m) in ph]
    return float(np.median(v)) if v else None

def mark_phase(ax, x, label=True, color="#00E5FF"):
    """Dashed vertical line at the start of the cuttlefish activity (x = None -> nothing)."""
    if x is None or not np.isfinite(x):
        return
    ax.axvline(x, color=color, lw=1.6, ls="--", zorder=6)
    if label:
        ax.text(x, 0.98, " cuttlefish active →", transform=ax.get_xaxis_transform(),
                color=color, fontsize=7, va="top", ha="left", zorder=7)

print(f"[OK] Config — SUBJECT={SUBJECT} | DATASET={DATASET} | {len(FEATURES)} features | "
      f"weights ≠ 1: { {f: w for f, w in FEATURE_WEIGHTS.items() if w != 1.0} or 'none'} | {len(SMOOTH_FILES)} file(s) | "
      f"1 frame / {FRAME_STEP} for Leiden | results → {OUT_DIR}")

[OK] Config — DATASET=cut_new | 6 features | weights ≠ 1: {'attack_side': 3.0, 'cuttle_approach_speed': 2.0, 'orbit_speed': 5.0, 'who_moves': 4.0} | 1 file(s) | 1 frame / 1 for Leiden | results → c:\Users\nonoh\Documents\01Travail\02_CREC\cuttle_strike\Dossier_final\Results\analysis_results_cut_new


In [4]:
# ── No display in this notebook: figures are never shown ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.show = lambda *a, **k: plt.close("all")
print("[OK] compute mode: figures disabled")

[OK] compute mode: figures disabled


## Frame clustering (Leiden) + UMAP

In [5]:
def run_leiden(df, features, k=100, res=1.0, k2=3, res2=5.0, seed=42,
               feature_weights=None):
    """
    Leiden level 1 (clusters) + level 2 (superclusters).

    feature_weights : optional dict {feature_name: weight}
        The weight is applied AFTER StandardScaler so that it is actually
        effective — multiplying by W gives W times more importance in
        the Euclidean distance between frames.
        E.g. : {"orientation_variation": 3.0} → this feature counts
             3× more than the others in the clustering.
    """

    # ── Level 1 — clusters on the frames ───────────────────
    X_raw = df[features].values.copy().astype(float)

    # StandardScaler FIRST — all features at std=1
    X = StandardScaler().fit_transform(X_raw)

    # Weights AFTER StandardScaler
    # Before scaling: different stds → multiplying has no uniform effect
    # After scaling: std=1 for all → multiplying by W gives exactly
    #                W times more importance in the Euclidean distance
    if feature_weights:
        for i, feat in enumerate(features):
            w = feature_weights.get(feat, 1.0)
            if w != 1.0:
                X[:, i] *= w
                print(f"  [WEIGHT] {feat} × {w}  (std : 1.0 → {w:.1f})")

    A         = kneighbors_graph(X, n_neighbors=k, mode='distance')
    A         = A.tocoo()
    keep      = A.data != 0                         # comme A.nonzero()
    s, t      = A.row[keep], A.col[keep]
    dist_vals = A.data[keep]
    w_vals    = 1 - dist_vals / (dist_vals.max() + 1e-9)

    # tableaux numpy (pas de liste Python de tuples) : même graphe, bien moins de mémoire
    g = ig.Graph(n=len(X), edges=np.column_stack([s, t]), directed=False)
    g.es["weight"] = w_vals

    p = leidenalg.find_partition(
        g, leidenalg.RBConfigurationVertexPartition,
        weights="weight",
        resolution_parameter=res, seed=seed
    )
    df            = df.copy()
    df["cluster"] = p.membership
    n_clusters    = df["cluster"].nunique()

    # ── Level 2 — superclusters on the centroids ───────────
    mean_c  = df.groupby("cluster")[features].mean()
    mean_cs = StandardScaler().fit_transform(mean_c)

    A2      = kneighbors_graph(mean_cs,
                                n_neighbors=min(k2, len(mean_c) - 1),
                                mode='distance')
    s2, t2  = A2.nonzero()
    dist2   = np.array(A2[s2, t2]).flatten()
    w2      = 1 - dist2 / (dist2.max() + 1e-9)

    g2 = ig.Graph(n=len(mean_c), edges=list(zip(s2, t2)), directed=False)
    g2.es["weight"] = w2.tolist()

    p2   = leidenalg.find_partition(
        g2, leidenalg.RBConfigurationVertexPartition,
        weights="weight",
        resolution_parameter=res2, seed=seed
    )
    c2sc               = {cid: sl for cid, sl in zip(mean_c.index, p2.membership)}
    df["supercluster"] = df["cluster"].map(c2sc)
    n_super            = df["supercluster"].nunique()

    print(f"  → {n_clusters} clusters | {n_super} superclusters")
    return df, X, n_clusters, n_super


def run_umap(X, seed=42):
    return umap.UMAP(n_components=2, random_state=seed).fit_transform(X)


# ── Feature weights ────────────────────────────────────────
# FEATURE_WEIGHTS is defined in CELL 0 depending on DATASET
# (FEATURE_WEIGHTS_OLD for "cut_old", FEATURE_WEIGHTS_NEW for "cut_new" / "full")

print("[OK] Functions defined")

[OK] Functions defined


In [6]:
results = {}

USE_CACHE = True   # False = force a full recomputation

def _cache_params(fpath):
    """Everything that, if it changes, invalidates the cache
    (features, weights, Leiden parameters, modified source CSV)."""
    st = Path(fpath).stat()
    return {"features": list(FEATURES), "weights": dict(FEATURE_WEIGHTS),
            "leiden": [LEIDEN_K, LEIDEN_RES, LEIDEN_K2, LEIDEN_RES2]
                      + ([FRAME_STEP] if FRAME_STEP > 1 else []),
            "exclude": sorted(EXCLUDE_ANIMALS),
            "orbit_min": ORBIT_MIN_DEG_S,
            "csv": [str(fpath), st.st_size, st.st_mtime]}

def _leiden_subsampled(df_w, step=FRAME_STEP, k_prop=15):
    """
    Leiden + UMAP on 1 frame out of `step` (per individual). Every other frame
    gets the MAJORITY cluster and the MEAN UMAP position of its k_prop
    nearest neighbours in feature space -> every frame gets
    a cluster, the sequences stay complete.
    """
    from sklearn.neighbors import NearestNeighbors
    df_w = df_w.sort_values(["animal_id", "frame"]).reset_index(drop=True)
    samp = (df_w.groupby("animal_id").cumcount() % step == 0).to_numpy()
    df_s, X_s, n_cl, n_su = run_leiden(
        df_w[samp].reset_index(drop=True), FEATURES,
        k=LEIDEN_K, res=LEIDEN_RES, k2=LEIDEN_K2, res2=LEIDEN_RES2,
        feature_weights=FEATURE_WEIGHTS)
    Z_s = run_umap(X_s)

    # same transform as run_leiden (StandardScaler of the subsample + weights)
    raw_s = df_w.loc[samp, FEATURES].to_numpy(float)
    mu, sd = raw_s.mean(0), raw_s.std(0)
    sd[sd == 0] = 1.0
    wts   = np.array([FEATURE_WEIGHTS.get(f, 1.0) for f in FEATURES])
    X_all = (df_w[FEATURES].to_numpy(float) - mu) / sd * wts

    # non-sampled frames: vote of the k_prop sampled neighbours
    lab_s = df_s["cluster"].to_numpy()
    nn    = NearestNeighbors(n_neighbors=k_prop).fit(X_s)
    other = ~samp
    _, idx = nn.kneighbors(X_all[other])
    counts = np.zeros((other.sum(), n_cl))
    for j in range(k_prop):
        counts[np.arange(other.sum()), lab_s[idx[:, j]]] += 1
    cluster = np.empty(len(df_w), dtype=int)
    cluster[samp], cluster[other] = lab_s, counts.argmax(1)
    Z = np.empty((len(df_w), 2))
    Z[samp], Z[other] = Z_s, Z_s[idx].mean(1)

    c2sc = df_s.groupby("cluster")["supercluster"].first()
    df_w["cluster"]        = cluster
    df_w["supercluster"]   = df_w["cluster"].map(c2sc)
    df_w["leiden_sampled"] = samp

    # check: on sampled frames, does the neighbours' vote (without
    # themselves) give back their true Leiden cluster?
    rng = np.random.default_rng(0)
    h   = rng.choice(len(X_s), min(5000, len(X_s)), replace=False)
    _, ih = nn.kneighbors(X_s[h], n_neighbors=k_prop + 1)
    vote = np.array([np.bincount(lab_s[r[1:]], minlength=n_cl).argmax() for r in ih])
    print(f"  [SUBSAMPLE] Leiden/UMAP on {samp.sum()} frames (1/{step}) | "
          f"{other.sum()} frames assigned by neighbours | vote consistency {np.mean(vote == lab_s[h]):.0%}")
    return df_w, X_all, Z, n_cl, n_su


for window, fpath in SMOOTH_FILES.items():
    print(f"\n── Smooth window={window} ──")
    cache  = OUT_DIR / f"smooth_{window}_results.pkl"
    params = _cache_params(fpath)

    # ── Try the cache ──
    if USE_CACHE and cache.exists():
        with open(cache, "rb") as f:
            res_w = pickle.load(f)
        if res_w.get("params") == params:
            results[window] = res_w
            print(f"  [CACHE] {cache.name} reloaded — "
                  f"{res_w['n_clusters']} clusters | {res_w['n_superclusters']} superclusters")
            continue
        print(f"  [CACHE] different parameters or old cache -> recomputing")

    df_w = pd.read_csv(fpath)
    df_w = df_w[~df_w["animal_id"].isin(EXCLUDE_ANIMALS)].reset_index(drop=True)
    # orbit: ORBIT_MIN_DEG_S threshold (cell 0) then log
    if "orbit_speed_deg_s" in df_w.columns and "orbit_speed" in FEATURES:
        df_w["orbit_speed"] = np.log1p(np.clip(df_w["orbit_speed_deg_s"] - ORBIT_MIN_DEG_S, 0, None))
    print(f"  {len(df_w)} frames | {df_w['animal_id'].nunique()} individuals")

    if FRAME_STEP > 1:
        df_w, X_w, Z_w, n_cl, n_su = _leiden_subsampled(df_w)
    else:
        df_w, X_w, n_cl, n_su = run_leiden(
            df_w, FEATURES,
            k=LEIDEN_K, res=LEIDEN_RES,
            k2=LEIDEN_K2, res2=LEIDEN_RES2,
            feature_weights=FEATURE_WEIGHTS    # ← added
        )
        Z_w = run_umap(X_w)

    res_w = {"df": df_w, "X": X_w, "Z": Z_w,
             "n_clusters": n_cl, "n_superclusters": n_su, "window": window,
             "features": list(FEATURES), "params": params}
    results[window] = res_w

    df_w.to_csv(OUT_DIR / f"smooth_{window}_clustered.csv", index=False)
    with open(cache, "wb") as f:
        pickle.dump(res_w, f)
    print(f"  [OK] → smooth_{window}_clustered.csv")

print("\n[OK] All files processed")


── Smooth window=1 ──
  [CACHE] different parameters or old cache -> recomputing
  90274 frames | 200 individuals
  [WEIGHT] attack_side × 3.0  (std : 1.0 → 3.0)
  [WEIGHT] cuttle_approach_speed × 2.0  (std : 1.0 → 2.0)
  [WEIGHT] orbit_speed × 5.0  (std : 1.0 → 5.0)
  [WEIGHT] who_moves × 4.0  (std : 1.0 → 4.0)
  → 23 clusters | 11 superclusters


c:\Users\nonoh\anaconda3\envs\cuttlestrike\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  [OK] → smooth_1_clustered.csv

[OK] All files processed


In [7]:
SELECTED_WINDOW = 1

In [8]:
# ── Working data of the selected window ──
R = results[SELECTED_WINDOW]
df              = R["df"].copy()
X, Z            = R["X"], R["Z"]
n_clusters      = R["n_clusters"]
n_superclusters = R["n_superclusters"]
animal_ids      = sorted(df["animal_id"].unique())
print(f"[OK] window {SELECTED_WINDOW}: {len(df)} frames | {n_clusters} clusters | "
      f"{n_superclusters} superclusters | {len(animal_ids)} individuals")

[OK] window 1: 90274 frames | 23 clusters | 11 superclusters | 200 individuals


## Strategies (SeqVec + Leiden kNN)

In [9]:
import io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import Button, Slider, TextBox
from collections import Counter
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler


# ── Sequence extraction ──────────────────────────────────
def _extract_sequences(df, cluster_col, min_run_len):
    sequences, mono_ids, multi_ids = {}, [], []
    for aid, group in df.groupby("animal_id"):
        raw = group.sort_values("frame")[cluster_col].values
        blocks, lengths = [], []
        cur, start = raw[0], 0
        for i in range(1, len(raw)):
            if raw[i] != cur:
                blocks.append(int(cur)); lengths.append(i - start)
                cur, start = raw[i], i
        blocks.append(int(cur)); lengths.append(len(raw) - start)
        fb, fl = [], []
        for b, l in zip(blocks, lengths):
            if l >= min_run_len:
                if fb and fb[-1] == b: fl[-1] += l
                else: fb.append(b); fl.append(l)
        sequences[aid] = {"seq": fb, "lengths": fl,
                          "total": len(raw)}       # real duration (frames), short segments included
        (mono_ids if len(set(fb)) <= 1 else multi_ids).append(aid)
    return sequences, mono_ids, multi_ids


# ── State similarity, alignment and dynamics ───────────────
try:
    from numba import njit
except ImportError:                       # no numba: pure Python version (slower)
    def njit(*a, **k):
        return a[0] if a and callable(a[0]) else (lambda f: f)


def _state_kernel_half(state_dist):
    """Square root of the state similarity K = exp(-d²/2σ²), σ = median(d)/2.
    v @ K½ makes the Euclidean distance "soft": two close states
    (neighbouring prototypes) count as similar."""
    d = np.asarray(state_dist, float)
    sigma = np.median(d[d > 0]) / 2 if (d > 0).any() else 1.0
    K = np.exp(-d ** 2 / (2 * sigma ** 2))
    ev, evec = np.linalg.eigh(K)
    return evec @ np.diag(np.sqrt(np.clip(ev, 0, None))) @ evec.T


@njit(cache=False)
def _align_cost(a, b, sub, gap):
    """Global alignment (Needleman-Wunsch) of two state sequences.
    Substitution = distance between prototypes (0..1), insertion/deletion = gap.
    Normalised by the length of the longest sequence."""
    n, m = len(a), len(b)
    prev = np.empty(m + 1)
    for j in range(m + 1):
        prev[j] = j * gap
    for i in range(1, n + 1):
        cur = np.empty(m + 1)
        cur[0] = i * gap
        for j in range(1, m + 1):
            c = prev[j - 1] + sub[a[i - 1], b[j - 1]]
            if prev[j] + gap < c:
                c = prev[j] + gap
            if cur[j - 1] + gap < c:
                c = cur[j - 1] + gap
            cur[j] = c
        prev = cur
    return prev[m] / max(n, m, 1)


@njit(cache=False)
def _dtw_cost(x, y, radius):
    """Multivariate DTW (time × features) with a Sakoe-Chiba band:
    compares how the features evolve while tolerating a slight time shift."""
    n, m = x.shape[0], y.shape[0]
    D = np.full((n + 1, m + 1), 1e18)
    D[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(max(1, i - radius), min(m, i + radius) + 1):
            d = 0.0
            for k in range(x.shape[1]):
                d += (x[i - 1, k] - y[j - 1, k]) ** 2
            best = D[i - 1, j - 1]
            if D[i - 1, j] < best:
                best = D[i - 1, j]
            if D[i, j - 1] < best:
                best = D[i, j - 1]
            D[i, j] = np.sqrt(d) + best
    return D[n, m]


def _pairwise(items, fn, *args):
    n = len(items)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            M[i, j] = M[j, i] = fn(items[i], items[j], *args)
    return M


# ── SeqVec ────────────────────────────────────────────────────
#   Each individual is described by 9 families, each answers one question:
#     hist     : how much time in each state?
#     temporal : which states in the 1st vs 2nd half (approach -> before the strike)?
#     lastapp  : which states just before the strike? (last appearance of each state)
#     duration : total duration of the sequence (1 column, log)
#     align    : is the order of the states similar? (close states = cheap substitution)
#     dynamics : do the features evolve the same way? (DTW on the features, without the states)
#     speed    : mean speed of the cuttlefish (1 column, log); with duration = distance travelled
#     path     : total distance travelled by the cuttlefish (mm, 1 column, log): did it move a lot?
#     start    : distance to the prey at the START of the sequence (mm, 1 column, log): does it start close?
#   Distance: D² = Σ_f w_f · (D_f / scale_f)² -> each family weighs w_f / Σw,
#   whatever its number of columns.
def _seqvec(sequences, ids, n_cl,
            w_hist=1.0, w_temporal=1.0, w_lastapp=1.0, w_duration=0.5, w_align=1.0,
            w_dynamics=1.0, w_speed=0.5, w_path=0.5, w_start=0.5, state_similarity=True,
            state_dist=None, align_gap=0.5, series=None, speed=None, path=None, start=None):

    def feat_hist(s, l, n):
        # share of time in each state (log1p: compresses extreme dominances)
        c = Counter()
        for cl, ln in zip(s, l):
            c[cl] += ln
        raw = np.array([c.get(i, 0) / sum(l) for i in range(n)])
        return np.log1p(raw * 10) / np.log1p(10)

    def feat_temporal(s, l, n):
        # share of time in each state in the 1st half (t1), 2nd half (t2)
        # and their difference (t2 - t1 > 0: state more present before the strike)
        total, half = sum(l), sum(l) / 2
        h1, h2, cum = np.zeros(n), np.zeros(n), 0
        for cl, ln in zip(s, l):
            if cl < n:
                in_h1 = max(0, min(ln, half - cum))
                h1[cl] += in_h1
                h2[cl] += ln - in_h1
            cum += ln
        t1, t2 = h1 / (half + 1e-9), h2 / (half + 1e-9)
        return np.concatenate([t1, t2, t2 - t1])

    def feat_lastapp(s, n):
        # presence (0/1) + position of the LAST appearance of each state
        # (1.0 = it is the final state, just before the strike)
        present, position = np.zeros(n), np.full(n, 0.5)
        for i in range(n):
            if i in s:
                present[i]  = 1.0
                position[i] = (len(s) - 1 - s[::-1].index(i)) / max(len(s) - 1, 1)
        return np.concatenate([present, position])

    # ── Raw vectors, grouped by family ────────────────────
    fam_sizes = {"hist": n_cl, "temporal": 3 * n_cl, "lastapp": 2 * n_cl, "duration": 1, "speed": 1, "path": 1,
                 "start": 1}
    weights   = {"hist": w_hist, "temporal": w_temporal, "lastapp": w_lastapp,
                 "duration": w_duration, "speed": w_speed if speed else 0.0,
                 "path": w_path if path else 0.0, "start": w_start if start else 0.0}
    blocks    = {f: [] for f in fam_sizes}
    for aid in ids:
        s, l = sequences[aid]["seq"], sequences[aid]["lengths"]
        if not s:                                   # empty sequence -> zero vector
            for f, k in fam_sizes.items():
                blocks[f].append(np.zeros(k))
            continue
        blocks["hist"].append(feat_hist(s, l, n_cl))
        blocks["temporal"].append(feat_temporal(s, l, n_cl))
        blocks["lastapp"].append(feat_lastapp(s, n_cl))
        # real duration of the sequence (not only the segments kept by min_run_len)
        blocks["duration"].append(np.array([np.log(sequences[aid].get("total", sum(l)))]))   # log: 100 -> 200 f = 1000 -> 2000 f
        blocks["speed"].append(np.array([np.log1p(speed.get(aid, 0.0)) if speed else 0.0]))   # mm/s, log
        blocks["path"].append(np.array([np.log1p(path.get(aid, 0.0)) if path else 0.0]))     # mm, log
        blocks["start"].append(np.array([np.log1p(start.get(aid, 0.0)) if start else 0.0]))   # mm, log
    blocks = {f: np.array(v, dtype=float) for f, v in blocks.items()}

    # ── Close states = similar ("soft" hist and temporal) ──
    if state_similarity and state_dist is not None:
        Kh = _state_kernel_half(state_dist)[:n_cl, :n_cl]
        blocks["hist"] = blocks["hist"] @ Kh
        blocks["temporal"] = (blocks["temporal"].reshape(len(ids), 3, n_cl) @ Kh
                              ).reshape(len(ids), 3 * n_cl)

    # ── Combination: each family brought to the same scale ─
    parts, used = [], {}
    for f in fam_sizes:
        if weights[f] <= 0:
            continue
        Xf = StandardScaler().fit_transform(blocks[f])
        scale = np.sqrt(np.mean(pdist(Xf) ** 2)) if len(Xf) > 1 else 0.0
        if scale < 1e-9:     # tolerance: a constant family varies by ~1e-15 (rounding)                              # constant family: adds nothing
            continue
        parts.append(Xf * np.sqrt(weights[f]) / scale)
        used[f] = weights[f]
    X  = np.hstack(parts) if parts else np.zeros((len(ids), 0))
    D2 = squareform(pdist(X)) ** 2 if parts else np.zeros((len(ids), len(ids)))

    # ── align: alignment distance of the state sequences ──────
    if w_align > 0:
        if state_dist is None:
            raise ValueError("w_align > 0 requires state_dist (provided by show_strategy_explorer)")
        sub = np.asarray(state_dist, float)[:n_cl, :n_cl]
        sub = sub / (sub.max() + 1e-12)
        sq  = [np.asarray(sequences[a]["seq"], dtype=np.int64) for a in ids]
        Da  = _pairwise(sq, _align_cost, sub, float(align_gap))
        sc  = np.sqrt(np.mean(Da[np.triu_indices(len(ids), 1)] ** 2))
        if sc > 1e-9:
            D2 += w_align * (Da / sc) ** 2
            used["align"] = w_align

    # ── dynamics: DTW on the feature series (N_DYN points per individual) ──
    if w_dynamics > 0:
        if series is None:
            raise ValueError("w_dynamics > 0 requires series (provided by show_strategy_explorer)")
        ser = [np.ascontiguousarray(series[a], dtype=float) for a in ids]
        rad = max(1, int(0.1 * ser[0].shape[0]))          # tolerated shift: 10 % of the time
        Dd  = _pairwise(ser, _dtw_cost, rad)
        sc  = np.sqrt(np.mean(Dd[np.triu_indices(len(ids), 1)] ** 2))
        if sc > 1e-9:
            D2 += w_dynamics * (Dd / sc) ** 2
            used["dynamics"] = w_dynamics

    if not used:
        raise ValueError("All SeqVec weights are 0 (or constant families).")
    D   = np.sqrt(D2)
    tot = sum(used.values())
    print("[SEQVEC] mean share of each family in D² : " +
          " | ".join(f"{f} {w / tot:.0%}" for f, w in used.items()))
    desc = f"SeqVec ({', '.join(used)})"
    print(f"[SEQVEC] {len(ids)} individuals | {desc}")
    return X, D, desc
# ── Hierarchical clustering ───────────────────────────────────
def _cluster_hier(D, ids, n_strat, linkage_method="ward"):
    Z      = linkage(squareform(np.clip(D, 0, None)), method=linkage_method)
    labels = fcluster(Z, n_strat, criterion="maxclust")
    n2l    = {n: str(i) for i, n in enumerate(sorted(set(labels)))}
    return {aid: n2l[lab] for aid, lab in zip(ids, labels)}, Z


def _ward_stability(D, k, linkage_method="ward", n_boot=30, seed=0):
    """Group stability: remove 20 % of the individuals, redo the
    clustering and measure the agreement (ARI) with the full clustering."""
    from sklearn.metrics import adjusted_rand_score
    cut = lambda M: fcluster(linkage(squareform(np.clip(M, 0, None), checks=False),
                                     method=linkage_method), k, criterion="maxclust")
    rng, full, aris = np.random.default_rng(seed), cut(D), []
    for _ in range(n_boot):
        sub = np.sort(rng.choice(len(D), int(.8 * len(D)), replace=False))
        aris.append(adjusted_rand_score(full[sub], cut(D[np.ix_(sub, sub)])))
    return float(np.mean(aris))


def _silhouette_optimal_k(D, k_range=range(2, 13), linkage_method="ward"):
    """K that maximises the silhouette of the hierarchical clustering on D.
    Silhouette: is each individual closer to its own group than to the
    neighbouring group? (1 = clean groups, 0 = overlap)."""
    from sklearn.metrics import silhouette_score
    Zl  = linkage(squareform(np.clip(D, 0, None), checks=False), method=linkage_method)
    res = {}
    print("[SILHOUETTE] K tested (Ward) :")
    for k in k_range:
        if k >= len(D):
            break
        lab = fcluster(Zl, k, criterion="maxclust")
        if len(set(lab)) < 2:
            continue
        res[k] = silhouette_score(D, lab, metric="precomputed")
        print("   K={k:<3} silhouette {s:+.3f} | smallest group {m}".format(k=k, s=res[k], m=int(np.bincount(lab)[1:].min())))
    k_best = max(res, key=res.get)
    print(f"[SILHOUETTE] K selected : {k_best}")
    return k_best, res


# ── Leiden to estimate the optimal K ────────────────────────────
def _leiden_optimal_k(D, ids, resolution_range=None):
    try:
        import igraph as ig
        import leidenalg
    except ImportError:
        print("[WARN] igraph/leidenalg not available")
        return None, {}
    n = len(ids)
    if n < 3: return None, {}
    D_max = D.max()
    if D_max == 0: return 1, {}
    sim       = 1 - D / D_max
    threshold = np.percentile(sim[sim > 0], 50)
    edges, weights = [], []
    for i in range(n):
        for j in range(i+1, n):
            if sim[i, j] > threshold:
                edges.append((i, j)); weights.append(float(sim[i, j]))
    if not edges: return 1, {}
    g = ig.Graph(n=n, edges=edges)
    g.es["weight"] = weights
    if resolution_range is None:
        resolution_range = [0.5, 0.8, 0.9, 1.0, 1.1, 1.2, 1.5, 2.0]
    results = {}
    for res in resolution_range:
        part = leidenalg.find_partition(
            g, leidenalg.RBConfigurationVertexPartition,
            weights="weight", resolution_parameter=res, seed=42)
        # standard modularity (comparable across resolutions, unlike
        # part.quality() which drops mechanically as the resolution rises)
        results[res] = {"k": len(set(part.membership)),
                        "modularity": g.modularity(part.membership, weights="weight")}
    ok = {r: v for r, v in results.items() if v["k"] >= 2}   # at least 2 groups
    if not ok: return 1, results
    best_res  = max(ok, key=lambda r: ok[r]["modularity"])
    k_optimal = results[best_res]["k"]
    print(f"\n[LEIDEN] optimal K = {k_optimal} (resolution={best_res:.2f})")
    for res, r in results.items():
        print(f"  res={res:.2f} → K={r['k']} modularity={r['modularity']:.3f}")
    return k_optimal, results


def _leiden_knn_groups(D, ids, knn_k=10, min_size=3,
                       resolutions=(0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5)):
    """Groups = Leiden communities on the graph of the knn_k nearest neighbours.
    knn_k = number of NEIGHBOURS per individual (NOT the number of groups):
    Leiden decides the number of groups itself (resolution with the best modularity).
    Groups with < min_size individuals are attached to the closest group.
    Tested (test_k_methods.py): best method on synthetic and real data."""
    import igraph as ig
    import leidenalg
    n   = len(ids)
    sim = 1 - D / (D.max() + 1e-12)
    nn  = np.argsort(D, axis=1)[:, 1:knn_k + 1]
    edges = sorted({(min(i, j), max(i, j)) for i in range(n) for j in nn[i]})
    w = [float(sim[i, j]) for i, j in edges]
    g = ig.Graph(n=n, edges=edges)
    best, best_q, best_res = None, -np.inf, None
    for res in resolutions:
        p = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition,
                                     weights=w, resolution_parameter=res, seed=42)
        mem = np.array(p.membership)
        if len(set(mem)) < 2:
            continue
        q = g.modularity(mem, weights=w)
        if q > best_q:
            best, best_q, best_res = mem, q, res
    if best is None:
        return {a: "0" for a in ids}, 1
    lab = best.copy()
    while True:                                  # attach the small groups
        sizes = pd.Series(lab).value_counts()
        small, big = sizes[sizes < min_size].index, sizes[sizes >= min_size].index
        if len(small) == 0 or len(big) == 0:
            break
        for i in np.where(np.isin(lab, small))[0]:
            lab[i] = min(big, key=lambda c: D[i, lab == c].mean())
    order = pd.Series(lab).value_counts().index          # 0 = largest group
    l2s   = {c: str(r) for r, c in enumerate(order)}
    print(f"[LEIDEN kNN] {knn_k} neighbours | resolution {best_res} | modularity {best_q:.3f} "
          f"| {len(order)} groups")
    return {a: l2s[c] for a, c in zip(ids, lab)}, len(order)


def _knn_stability(D, knn_k=10, n_boot=30, seed=0):
    """Stability: remove 20 % of the individuals, redo the kNN groups and
    measure the agreement (ARI) with the full groups (> 0.75 good, < 0.5 fragile)."""
    from sklearn.metrics import adjusted_rand_score
    ids = list(range(len(D)))
    with contextlib.redirect_stdout(io.StringIO()):
        full = _leiden_knn_groups(D, ids, knn_k)[0]
        rng, aris = np.random.default_rng(seed), []
        for _ in range(n_boot):
            sub = np.sort(rng.choice(len(D), int(.8 * len(D)), replace=False))
            part = _leiden_knn_groups(D[np.ix_(sub, sub)], list(sub), knn_k)[0]
            aris.append(adjusted_rand_score([full[i] for i in sub], [part[i] for i in sub]))
    return float(np.mean(aris))


# ── Main window ────────────────────────────────────────
def show_strategy_explorer(
    df,
    cluster_col    = "cluster",
    n_strategies   = 4,
    min_run_len      = 3,
    use_leiden_k   = False,
    k_method       = None,            # "knn" (recommended) | "leiden" | "silhouette" | a number
    knn_k          = 10,              # k_method='knn': number of NEIGHBOURS per individual (not the number of groups)
    k_range        = range(2, 13),    # K tested by k_method='silhouette'
    linkage_method = "ward",
    # ── SeqVec: weights of the 9 families (see _seqvec) ──
    w_hist     = 1.0,     # time spent in each state
    w_temporal = 1.0,     # states at the start vs end of the sequence
    w_lastapp  = 1.0,     # states just before the strike
    w_duration = 0.5,     # total duration (1 column)
    w_align    = 1.0,     # order of the states (alignment)
    w_dynamics = 1.0,     # evolution of the features (DTW), without the states
    w_speed    = 0.5,     # mean speed (1 column); with duration = distance travelled
    w_path     = 0.5,     # total distance travelled (1 column)
    w_start    = 0.5,     # start distance to the prey (1 column): separates close-range strikes
    state_similarity = True,   # close states = similar
    features   = None,
):
    df = df.copy()
    df["animal_id"] = df["animal_id"].astype(str)

    # ── SeqVec inputs: state prototypes + feature series ──
    feats  = list(features) if features is not None else list(FEATURES)
    fw     = globals().get("FEATURE_WEIGHTS", {}) or {}
    zs_all = pd.DataFrame(((df[feats] - df[feats].mean()) / df[feats].std()).values
                          * np.array([fw.get(f, 1.0) for f in feats]), index=df.index)
    seqvec_inputs = {}
    N_DYN = 50          # points per individual for the DTW (duration is in the duration family)

    def _seqvec_inputs(col, n_cl):
        # state prototype = centroid in the standardised + weighted space
        cent = zs_all.groupby(df[col].values).mean().reindex(range(n_cl)).fillna(0).values
        # series of each individual: N_DYN successive means (normalised time)
        ser = {}
        for aid, g in df.groupby("animal_id"):
            v = zs_all.loc[g.sort_values("frame").index].values
            e = np.linspace(0, len(v), N_DYN + 1).astype(int)
            ser[aid] = np.vstack([v[e[i]:max(e[i + 1], e[i] + 1)].mean(0) for i in range(N_DYN)])
        # mean speed of each individual (speed family)
        spd = (df.groupby("animal_id")[SPEED_COL].mean().to_dict()
               if SPEED_COL in df.columns else None)
        # total distance travelled by each individual (path family, mm)
        pth = ((df.groupby("animal_id")[SPEED_COL].sum() / 60).to_dict()
               if SPEED_COL in df.columns else None)
        # distance to the prey at the start of the sequence (start family, mm)
        st0 = (df.sort_values("frame").groupby("animal_id")["distance_mm"].first().to_dict()
               if "distance_mm" in df.columns else None)
        return {"state_dist": squareform(pdist(cent)), "state_similarity": state_similarity,
                "series": ser,
                "speed": spd,
                "path": pth,
                "start": st0}

    def compute_all(col, n_s):
        n_cl = int(df[col].nunique())
        seqs, mono, multi = _extract_sequences(df, col, min_run_len)
        print(f"[INFO] {len(multi)} multi | {len(mono)} mono | n_cl={n_cl}")
        if len(multi) < 2:
            return seqs, mono, multi, None, None, {}, n_cl, "—", n_s
        seqvec_inputs.clear()
        seqvec_inputs.update(_seqvec_inputs(col, n_cl))
        X, D, desc = _seqvec(
            seqs, multi, n_cl,
            w_hist=w_hist, w_temporal=w_temporal, w_lastapp=w_lastapp,
            w_duration=w_duration, w_align=w_align, w_dynamics=w_dynamics,
            w_speed=w_speed, w_path=w_path, w_start=w_start, **seqvec_inputs,
        )
        method = (k_method if k_method is not None
                  else ("leiden" if use_leiden_k else n_s))
        if method == "knn":
            strat, n_s = _leiden_knn_groups(D, multi, knn_k)
            Z    = linkage(squareform(np.clip(D, 0, None), checks=False), method=linkage_method)  # dendrogram
            stab = _knn_stability(D, knn_k)
            print(f"[STABILITY] K={n_s} : bootstrap ARI {stab:.2f} (> 0.75 good, < 0.5 fragile)")
            print(f"[CLUSTER] {Counter(strat.values())} | {desc}")
            return seqs, mono, multi, D, Z, strat, n_cl, desc, n_s
        if method == "leiden":
            k_opt, _ = _leiden_optimal_k(D, multi)
            if k_opt is not None:
                n_s = k_opt
                print(f"[LEIDEN] K selected : {n_s}")
        elif method == "silhouette":
            n_s, _ = _silhouette_optimal_k(D, k_range, linkage_method)
        else:
            try:
                n_s = int(method)
            except (TypeError, ValueError):
                raise ValueError(f"k_method must be 'knn', 'leiden', 'silhouette' or a number (got {k_method!r})")
            if k_method is not None:
                print(f"[K] set by hand : {n_s}")
        strat, Z = _cluster_hier(D, multi, n_s, linkage_method)
        stab = _ward_stability(D, n_s, linkage_method)
        print(f"[STABILITY] K={n_s} : bootstrap ARI {stab:.2f} (> 0.75 good, < 0.5 fragile)")
        print(f"[CLUSTER] {Counter(strat.values())} | {desc}")
        return seqs, mono, multi, D, Z, strat, n_cl, desc, n_s

    seqs, mono, multi, D, Z, strat, n_cl, desc, n_s_eff = compute_all(
        cluster_col, n_strategies)

    cur = {"col": cluster_col, "n_s": n_s_eff,
           "seqs": seqs, "mono": mono, "multi": multi,
           "D": D, "Z": Z, "strat": strat, "n_cl": n_cl, "desc": desc}

    # ── Figure ────────────────────────────────────────────────
    fig = plt.figure(figsize=(24, 12))
    fig.patch.set_facecolor("#1a1a2e")

    ax_dend = fig.add_axes([0.02, 0.22, 0.19, 0.72])
    ax_heat = fig.add_axes([0.23, 0.22, 0.19, 0.72])
    ax_seq  = fig.add_axes([0.45, 0.60, 0.53, 0.34])
    zone_r  = [0.45, 0.22, 0.53, 0.34]

    for ax in [ax_dend, ax_heat, ax_seq]:
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#555555", labelsize=7)

    ax_s_k   = fig.add_axes([0.02, 0.11, 0.19, 0.03])
    ax_info  = fig.add_axes([0.02, 0.03, 0.60, 0.05])
    ax_tb    = fig.add_axes([0.66, 0.11, 0.14, 0.04])
    ax_bshow = fig.add_axes([0.81, 0.11, 0.08, 0.04])
    ax_bleid = fig.add_axes([0.66, 0.04, 0.23, 0.04])

    slider   = Slider(ax_s_k, "K groups", 2,
                      max(3, min(12, len(multi)-1) if len(multi) > 2 else 12),
                      valinit=n_s_eff, valstep=1, color="#4466aa")
    textbox  = TextBox(ax_tb,    "Animal ID: ", initial="",
                       color="#111122", hovercolor="#1a1a2e")
    btn_show = Button(ax_bshow, "Show",  color="#224422", hovercolor="#336633")
    btn_leid = Button(ax_bleid, "Leiden kNN groups",
                      color="#442244", hovercolor="#663366")

    for btn in [btn_show, btn_leid]:
        btn.label.set_color("#cccccc"); btn.label.set_fontsize(9)
    slider.label.set_color("white"); slider.valtext.set_color("white")
    textbox.label.set_color("#aaaaaa"); textbox.text_disp.set_color("#ffffff")

    ax_info.axis("off")
    info_txt = ax_info.text(
        0, 0.5,
        f"col={cluster_col} | seqvec | min_run_len={min_run_len} | "
        f"K={n_s_eff} | {len(multi)} multi + {len(mono)} MONO",
        color="#888888", fontsize=8, va="center",
        transform=ax_info.transAxes, fontfamily="monospace")

    radar_axes = []

    def remove_radars():
        for ax in radar_axes: ax.remove()
        radar_axes.clear()

    def update_info():
        info_txt.set_text(
            f"col={cur['col']} | seqvec | min_run_len={min_run_len} | "
            f"K={cur['n_s']} | {len(cur['multi'])} multi + {len(cur['mono'])} MONO")

    def draw():
        seqs  = cur["seqs"]; mono  = cur["mono"]; multi = cur["multi"]
        D     = cur["D"];    Z     = cur["Z"];    strat = cur["strat"]
        n_cl  = cur["n_cl"]; n_s   = cur["n_s"]

        strategy_list = sorted(set(strat.values())) if strat else []
        pal = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_list)}
        pal["MONO"] = "#666666"

        # ── Dendrogram ──────────────────────────────────────
        ax_dend.cla(); ax_dend.set_facecolor("#0d0d1a")
        for sp in ax_dend.spines.values(): sp.set_edgecolor("#333333")
        ax_dend.tick_params(colors="#555555", labelsize=6)
        if Z is not None and len(multi) > 2:
            cut_h = Z[-(n_s-1), 2]*1.01 if len(Z) >= n_s else Z[-1, 2]
            dendrogram(Z, ax=ax_dend, no_labels=True,
                       color_threshold=cut_h,
                       above_threshold_color="#444455")
            ax_dend.axhline(cut_h, color="#FFD700", lw=1.2, ls="--",
                            label=f"K={n_s}")
            ax_dend.legend(fontsize=7, labelcolor="white",
                           loc="upper right").get_frame().set_facecolor("#111111")
        ax_dend.set_title(
            f"Dendrogram\n{len(multi)} multi + {len(mono)} MONO",
            color="#aaaaaa", fontsize=8, pad=6)
        ax_dend.set_xlabel("Individuals", color="#555555", fontsize=7)
        ax_dend.set_ylabel("Distance",  color="#555555", fontsize=7)

        # ── Heatmap ───────────────────────────────────────────
        ax_heat.cla(); ax_heat.set_facecolor("#0d0d1a")
        for sp in ax_heat.spines.values(): sp.set_edgecolor("#333333")
        if D is not None and len(multi) > 0:
            sorted_ids = sorted(multi, key=lambda x: strat.get(x, "?"))
            idx_map    = {aid: i for i, aid in enumerate(multi)}
            order      = [idx_map[aid] for aid in sorted_ids]
            Ds         = D[np.ix_(order, order)]
            im = ax_heat.imshow(Ds, cmap="viridis", aspect="auto")
            ax_heat.set_xticks([]); ax_heat.set_yticks([])
            prev_s, xp = None, 0
            for aid in sorted_ids:
                s = strat.get(aid, "?")
                if s != prev_s and prev_s is not None:
                    ax_heat.axhline(xp-.5, color="white", lw=0.8, alpha=0.5)
                    ax_heat.axvline(xp-.5, color="white", lw=0.8, alpha=0.5)
                prev_s = s; xp += 1
            plt.colorbar(im, ax=ax_heat, fraction=0.03, pad=0.02,
                         label="Dist.").ax.tick_params(colors="white", labelsize=5)
        patches = ([mpatches.Patch(color=pal[s],
                    label=f"{s} (n={sum(1 for v in strat.values() if v==s)})")
                    for s in strategy_list]
                   + [mpatches.Patch(color="#666666",
                      label=f"MONO (n={len(mono)})")])
        ax_heat.legend(handles=patches, fontsize=6, loc="lower right",
                       framealpha=0.5).get_frame().set_facecolor("#111111")
        ax_heat.set_title(f"Distances\n{cur['desc'][:35]}",
                          color="#aaaaaa", fontsize=8, pad=6)

        # ── Radars ────────────────────────────────────────────
        remove_radars()
        all_s = strategy_list + (["MONO"] if mono else [])
        nc = min(5, len(all_s)); nr = (len(all_s)+nc-1)//nc
        w_r = zone_r[2]/nc; h_r = zone_r[3]/nr

        for si, s in enumerate(all_s):
            members = mono if s == "MONO" else \
                      [aid for aid, lab in strat.items() if lab == s]
            profs = []
            for aid in members:
                sq = seqs[aid]["seq"]; lng = seqs[aid]["lengths"]
                total = sum(lng)
                if total == 0: continue
                h = np.zeros(n_cl)
                for cl, l in zip(sq, lng):
                    if cl < n_cl: h[cl] += l / total
                profs.append(h)
            if not profs: continue
            mp = np.mean(profs, axis=0)
            ci = si % nc; ri = si // nc
            rect = [zone_r[0]+ci*w_r, zone_r[1]+(nr-1-ri)*h_r,
                    w_r*0.90, h_r*0.88]
            ax_r = fig.add_axes(rect, polar=True)
            ax_r.set_facecolor("#0d0d1a"); radar_axes.append(ax_r)
            angs = np.linspace(0, 2*np.pi, n_cl, endpoint=False)
            af   = np.concatenate([angs, [angs[0]]])
            vf   = np.concatenate([mp, [mp[0]]])
            col  = pal.get(s, "#888888")
            ax_r.plot(af, vf, lw=2, color=col)
            ax_r.fill(af, vf, alpha=0.3, color=col)
            ax_r.set_xticks(angs)
            ax_r.set_xticklabels([f"C{c}" for c in range(n_cl)],
                                  fontsize=4, color="#aaaaaa")
            ax_r.set_ylim(0, max(mp.max()*1.2, 0.01))
            ax_r.set_yticks([])
            ax_r.grid(color="#333333", lw=0.5)
            ax_r.spines["polar"].set_color("#333333")
            ax_r.set_title(f"{s} n={len(members)}",
                           color=col, fontsize=7, pad=4)

        fig.suptitle(
            f"Strategies | col={cur['col']} | SeqVec+ | "
            f"K={n_s} | {len(multi)} multi + {len(mono)} MONO",
            color="#cccccc", fontsize=10, y=0.99)
        update_info()
        fig.canvas.draw_idle()

    draw()

    # ── Individual timeline ─────────────────────────────────────
    def show_indiv(event=None):
        raw   = textbox.text.strip()
        seqs  = cur["seqs"]; strat = cur["strat"]
        strategy_list = sorted(set(strat.values())) if strat else []
        pal = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
               for i, s in enumerate(strategy_list)}
        pal["MONO"] = "#666666"

        aid = str(raw) if raw in seqs else None
        if aid is None and raw.isdigit():
            aid_int = str(int(raw))
            if aid_int in seqs: aid = aid_int

        ax_seq.cla(); ax_seq.set_facecolor("#0d0d1a")
        for sp in ax_seq.spines.values(): sp.set_edgecolor("#333333")

        if aid is None:
            sample = list(seqs.keys())[:8]
            ax_seq.text(0.5, 0.5,
                f"'{raw}' not found\nExamples : {sample}",
                ha="center", va="center", color="#ff4444",
                transform=ax_seq.transAxes, fontsize=9)
            fig.canvas.draw_idle(); return

        sq  = seqs[aid]["seq"]; lng = seqs[aid]["lengths"]
        s   = strat.get(aid, "MONO"); col = pal.get(s, "#888888")
        ax_seq.set_title(
            f"Animal {aid}  —  Strategy {s}  |  "
            f"{len(sq)} blocks, {sum(lng)} frames",
            color=col, fontsize=9)
        ax_seq.set_yticks([])
        ax_seq.tick_params(colors="#555555", labelsize=7)

        x = 0
        for cluster, length in zip(sq, lng):
            c = PALETTE[cluster % len(PALETTE)]
            ax_seq.barh(0, length, left=x, height=0.6,
                        color=c, edgecolor="#111122", lw=0.4)
            if length >= max(lng) * 0.06:
                ax_seq.text(x + length/2, 0, str(cluster),
                            ha="center", va="center",
                            fontsize=7, color="white", fontweight="bold")
            x += length

        ax_seq.set_xlim(0, x); ax_seq.set_ylim(-0.5, 0.5)
        ax_seq.set_xlabel("Frames", color="#555555", fontsize=7)
        ax_seq.grid(axis="x", alpha=0.15, color="#334455")
        patches = [mpatches.Patch(color=PALETTE[c % len(PALETTE)],
                   label=f"C{c}") for c in sorted(set(sq))]
        ax_seq.legend(handles=patches, fontsize=6, loc="upper right",
                      framealpha=0.5).get_frame().set_facecolor("#111111")
        fig.canvas.draw_idle()

    # ── Events ────────────────────────────────────────────
    def on_leiden(event):
        if cur["D"] is None: return
        print("[LEIDEN kNN] Computing...")
        strat, k_opt = _leiden_knn_groups(cur["D"], cur["multi"], knn_k)
        cur["n_s"] = k_opt
        slider.eventson = False; slider.set_val(min(k_opt, slider.valmax)); slider.eventson = True
        cur.update({"strat": strat})
        remove_radars(); draw()

    def on_slider(val):
        cur["n_s"] = int(val)
        if cur["D"] is not None:
            strat, Z = _cluster_hier(cur["D"], cur["multi"],
                                      cur["n_s"], linkage_method)
            cur.update({"strat": strat, "Z": Z})
        remove_radars(); draw()

    slider.on_changed(on_slider)
    btn_show.on_clicked(show_indiv)
    btn_leid.on_clicked(on_leiden)
    textbox.on_submit(lambda t: show_indiv())

    plt.show()

    # ── Return final_state ─────────────────────────────────
    # ── Return final_state ─────────────────────────────────
    strat_final = cur["strat"] if cur["strat"] else {}
    mono_final  = cur["mono"]

    # Convert numbers → letters (A, B, C...)
    unique_groups   = sorted(set(v for v in strat_final.values()))
    num_to_letter   = {g: chr(65+i) for i, g in enumerate(unique_groups)}
    mapping_letters = {aid: num_to_letter.get(g, "MONO")
                       for aid, g in strat_final.items()}
    for m in mono_final:
        mapping_letters[str(m)] = "MONO"

    return {
        "seqs"           : cur["seqs"],
        "strat"          : strat_final,
        "mono"           : mono_final,
        "multi"          : cur["multi"],
        "n_cl"           : cur["n_cl"],
        "col"            : cur["col"],   # ← added
        "D"              : cur["D"],
        "Z"              : cur["Z"],
        "n_s"            : cur["n_s"],
        "desc"           : cur["desc"],
        "mapping_letters": mapping_letters,
        "seqvec_inputs"  : seqvec_inputs,
        "seqvec_weights" : dict(w_hist=w_hist, w_temporal=w_temporal, w_lastapp=w_lastapp,
                                w_duration=w_duration, w_align=w_align, w_dynamics=w_dynamics,
                                w_speed=w_speed, w_path=w_path, w_start=w_start),
        "knn_k"          : knn_k,
        "min_run_len"      : min_run_len,
        }

In [ ]:
# ── Call ─────────────────────────────────────────────────────
final_state = show_strategy_explorer(
    df             = df,
    cluster_col    = "supercluster",
    n_strategies   = 4,
    min_run_len      = 15,    # segments < 15 frames (0.25 s) ignored: removes state flickering
    use_leiden_k   = True,
    k_method       = "knn",          # "knn" (Leiden on kNN graph, recommended) | "leiden" | "silhouette" | a number
    knn_k          = 10,             # number of NEIGHBOURS per individual in the graph (NOT the number of groups)
    linkage_method = "ward",
    # ── SeqVec: weights of the 9 families ──
    w_hist         = 1.0,     # time spent in each state
    w_temporal     = 1.0,     # states at the start vs end of the sequence
    w_lastapp      = 1.0,     # states just before the strike
    w_duration     = 0.5,     # total duration (1 column)
    w_align        = 1.0,     # order of the states (alignment, close states = low cost)
    w_dynamics     = 1.0,     # evolution of the features (DTW), without the states
    w_speed        = 0.5,     # mean speed of the cuttlefish (with duration = distance travelled)
    w_path         = 1.0,     # total distance travelled by the cuttlefish (did it move a lot?)
    w_start        = 1.0,     # start distance to the prey (does it start close?)
    state_similarity = True,  # close states = similar (hist, temporal)
)

[INFO] 183 multi | 17 mono | n_cl=23
[SEQVEC] mean share of each family in D² : hist 12% | temporal 12% | lastapp 12% | duration 6% | speed 6% | path 12% | start 12% | align 12% | dynamics 12%
[SEQVEC] 183 individuals | SeqVec (hist, temporal, lastapp, duration, speed, path, start, align, dynamics)
[LEIDEN kNN] 10 neighbours | resolution 0.8 | modularity 0.564 | 6 groups
[STABILITY] K=6 : bootstrap ARI 0.71 (> 0.75 good, < 0.5 fragile)
[CLUSTER] Counter({'0': 44, '1': 32, '2': 31, '3': 27, '4': 26, '5': 23}) | SeqVec (hist, temporal, lastapp, duration, speed, path, start, align, dynamics)


## Optimal regrouping

In [11]:
# CELL — show_power_analysis + optimal_regrouping
#  Computations on the SeqVec features (consistent with the clustering)
# ══════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import mannwhitneyu
from itertools import combinations
from collections import Counter


def build_seqvec_features_per_individual(final_state, normalize=False):
    """
    Rebuilds the SeqVec features for each individual.
    Used for the statistical tests → normalize=False (raw values).
    Consistent with the _seqvec used for clustering.
    """
    seqs  = final_state["seqs"]
    n_cl  = final_state["n_cl"]
    ids   = list(seqs.keys())
    rows  = []

    for aid in ids:
        sq, lg = seqs[aid]["seq"], seqs[aid]["lengths"]
        total  = sum(lg)
        if total == 0: continue

        # Proportions
        hist = np.zeros(n_cl)
        for c, l in zip(sq, lg):
            if c < n_cl: hist[c] += l / total

        # Transitions
        trans = np.zeros((n_cl, n_cl))
        for a, b in zip(sq[:-1], sq[1:]):
            if 0 <= a < n_cl and 0 <= b < n_cl:
                trans[a, b] += 1
        rs   = trans.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1
        trans_norm = (trans / rs).flatten()

        # Median durations
        durations = {c: [] for c in range(n_cl)}
        for c, l in zip(sq, lg):
            if c < n_cl: durations[c].append(l)
        runlen = np.array([np.median(durations[c]) if durations[c] else 0
                           for c in range(n_cl)])

        # First appearance
        firstapp = []
        for c in range(n_cl):
            try:    firstapp.append(sq.index(c) / len(sq))
            except: firstapp.append(1.0)
        firstapp = np.array(firstapp)

        # Global metrics
        n_used   = len(set(sq))
        entropy  = 0.0
        h_vals   = hist[hist > 0]
        if len(h_vals) > 1:
            entropy = -np.sum(h_vals * np.log(h_vals)) / np.log(n_cl)

        row = {}
        for c in range(n_cl):
            row[f"hist_C{c}"]     = hist[c]
            row[f"runlen_C{c}"]   = runlen[c]
            row[f"firstapp_C{c}"] = firstapp[c]
        for i in range(n_cl):
            for j in range(n_cl):
                row[f"trans_{i}_{j}"] = trans_norm[i * n_cl + j]
        row["entropy"]      = entropy
        row["n_cl_used"]    = n_used / n_cl
        row["inertia"]      = hist.max()
        row["total_frames"] = total
        rows.append({"animal_id": str(aid), **row})

    df_seq = pd.DataFrame(rows).set_index("animal_id")
    df_seq = df_seq.fillna(0)

    if normalize:
        from sklearn.preprocessing import StandardScaler
        df_seq = pd.DataFrame(
            StandardScaler().fit_transform(df_seq.values),
            index=df_seq.index, columns=df_seq.columns)

    print(f"[SEQ FEATURES] {df_seq.shape[0]} individuals × "
          f"{df_seq.shape[1]} features (raw)")
    return df_seq


def _bh_correction(p_values, alpha=0.05):
    """
    Benjamini-Hochberg correction (FDR).
    Returns a boolean array: True = significant after correction.
    """
    p   = np.array(p_values)
    n   = len(p)
    if n == 0: return np.array([], dtype=bool)
    ranked  = np.argsort(p)
    thresholds  = (np.arange(1, n+1) / n) * alpha
    passed  = p[ranked] < thresholds
    sig     = np.zeros(n, dtype=bool)
    if passed.any():
        last = np.where(passed)[0][-1]
        sig[ranked[:last+1]] = True
    return sig


def show_power_analysis(final_state, df,
                        features         = None,
                        use_raw_features = False,
                        top_pct_features = 1.0,    # 1.0 = keep all
                        alpha            = 0.05,
                        power_target     = 0.80):
    """
    Statistical power analysis.

    use_raw_features : True  → Cohen's d on raw behavioural features
                               (distance, speed...) — more interpretable
                       False → Cohen's d on SeqVec features (default)

    top_pct_features : proportion of the most discriminant features
                       to keep for the Cohen's d computation.
                       1.0 = all (default)
                       0.2 = top 20% most discriminant
                       → useful to reduce the noise of weakly
                         informative features that pull the median down
    """
    strat = final_state["strat"]
    mono  = final_state["mono"]

    # ── Feature choice ────────────────────────────────────
    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    if use_raw_features and features is not None:
        # Raw behavioural features — mean per individual
        # More interpretable biologically, Cohen's d naturally
        # larger because the features are on real scales
        df_indiv  = df_w.groupby("animal_id")[features].mean()
        feat_cols = list(features)
        feat_label = f"raw features ({len(feat_cols)})"
    else:
        # SeqVec features — 184 abstract features
        df_indiv  = build_seqvec_features_per_individual(
                        final_state, normalize=False)
        feat_cols = list(df_indiv.columns)
        feat_label = f"raw SeqVec ({len(feat_cols)} features)"

    df_indiv["group"] = [strat.get(str(aid), "MONO")
                          for aid in df_indiv.index]
    df_indiv = df_indiv.dropna(subset=feat_cols)

    strategy_list = sorted(set(strat.values()))
    all_groups    = strategy_list + (["MONO"] if mono else [])

    # ── Discriminant feature filter ────────────────────────
    # Keeps only the top_pct_features% most discriminant ones
    # Criterion: ratio between-group variance / total variance (η²)
    # → prevents non-discriminant features from pulling Cohen's d
    #   down and underestimating the power
    if top_pct_features < 1.0 and len(feat_cols) > 5:
        scores = []
        grand_mean = df_indiv[feat_cols].mean()
        for feat in feat_cols:
            var_total = df_indiv[feat].var()
            if var_total < 1e-10:
                scores.append(0); continue
            var_inter = 0
            for g in strategy_list:
                mask = df_indiv["group"] == g
                if mask.sum() < 2: continue
                ng = mask.sum()
                mg = df_indiv.loc[mask, feat].mean()
                var_inter += ng * (mg - grand_mean[feat])**2
            var_inter /= max(len(df_indiv) - 1, 1)
            scores.append(var_inter / (var_total + 1e-10))

        threshold = np.percentile(scores, (1 - top_pct_features) * 100)
        feat_cols = [f for f, s in zip(feat_cols, scores)
                     if s >= threshold]
        feat_label += f" → top {int(top_pct_features*100)}%"
        print(f"[FILTER] {len(feat_cols)} features kept "
              f"(top {int(top_pct_features*100)}% discriminant)")

    pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                   for i, s in enumerate(all_groups)}
    pal["MONO"] = "#666666"

    # ── Theoretical minimum N ──────────────────────────────────────
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power_target)

    def n_min_for_d(d):
        if d <= 0: return np.inf
        return int(np.ceil((z_alpha + z_beta)**2 / (2 * d**2)))

    n_min_large  = n_min_for_d(0.8)
    n_min_medium = n_min_for_d(0.5)
    n_min_small  = n_min_for_d(0.2)

    group_sizes = {g: int((df_indiv["group"] == g).sum())
                   for g in all_groups}

    # ── Pairwise computation with BH ─────────────────────────────────
    pairs        = list(combinations(strategy_list, 2))
    pair_results = []

    for g1, g2 in pairs:
        mask1    = df_indiv["group"] == g1
        mask2    = df_indiv["group"] == g2
        n1, n2   = mask1.sum(), mask2.sum()

        if n1 < 2 or n2 < 2:
            pair_results.append({
                "g1": g1, "g2": g2, "n1": n1, "n2": n2,
                "cohens_d": 0, "power_obs": 0,
                "p_value": 1.0, "significant": False,
                "n_min_needed": np.inf,
            }); continue

        ds, ps = [], []
        for feat in feat_cols:
            x1 = df_indiv.loc[mask1, feat].dropna().values
            x2 = df_indiv.loc[mask2, feat].dropna().values
            if len(x1) < 2 or len(x2) < 2: continue
            ps_std = np.sqrt((x1.std()**2 + x2.std()**2) / 2)
            ds.append(abs(x1.mean() - x2.mean()) / (ps_std + 1e-9))
            try:
                _, p = mannwhitneyu(x1, x2, alternative="two-sided")
                ps.append(p)
            except:
                ps.append(1.0)

        mean_d = float(np.median(ds)) if ds else 0.0
        p_min  = float(np.min(ps))    if ps else 1.0
        n_obs  = min(n1, n2)
        n_need = n_min_for_d(mean_d)

        if n_need > 0 and n_need < np.inf:
            ratio     = n_obs / n_need
            power_obs = float(np.clip(stats.norm.cdf(
                z_beta * np.sqrt(ratio)
                - z_alpha * (1/np.sqrt(ratio) - 1)), 0, 1))
        else:
            power_obs = 0.0

        pair_results.append({
            "g1": g1, "g2": g2, "n1": n1, "n2": n2,
            "cohens_d":    round(mean_d, 3),
            "power_obs":   round(power_obs, 3),
            "p_value":     round(p_min, 4),
            "significant": False,
            "n_min_needed": n_need,
        })

    df_pairs = pd.DataFrame(pair_results)

    # BH correction
    if len(df_pairs) > 0:
        sig_bh = _bh_correction(df_pairs["p_value"].values, alpha)
        df_pairs["significant"] = sig_bh
        print(f"[BH] {sig_bh.sum()}/{len(sig_bh)} significant pairs "
              f"after Benjamini-Hochberg correction")

    # ── Scores ────────────────────────────────────────────────
    n_tot        = len(strategy_list)
    n_ok         = sum(1 for g, n in group_sizes.items()
                       if g != "MONO" and n >= n_min_medium)
    score_size   = n_ok / n_tot if n_tot > 0 else 0
    score_power  = df_pairs["power_obs"].mean()   if len(df_pairs) > 0 else 0
    score_sig    = df_pairs["significant"].mean() if len(df_pairs) > 0 else 0
    score_global = score_power * 0.5 + score_sig * 0.3 + score_size * 0.2

    pair_labels  = [f"{r.g1} ↔ {r.g2}" for r in df_pairs.itertuples()]

    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Group sizes
    # ════════════════════════════════════════════════════════
    groups_plot = [g for g in all_groups if g != "MONO"]
    sizes_plot  = [group_sizes[g] for g in groups_plot]

    fig1, ax = plt.subplots(figsize=(max(8, len(groups_plot)*0.6), 5))
    fig1.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
    bars = ax.bar(groups_plot, sizes_plot,
                  color=[pal[g] for g in groups_plot],
                  edgecolor="#222", linewidth=0.8, alpha=0.9)
    for bar, n in zip(bars, sizes_plot):
        ax.text(bar.get_x()+bar.get_width()/2, n+0.05, str(n),
                ha="center", va="bottom", color="white",
                fontsize=9, fontweight="bold")
    ymax = max(sizes_plot + [n_min_small]) * 1.25
    ax.axhline(n_min_large,  color="#44ff44", lw=1.5, ls="--",
               label=f"Large effect d=0.8 → n≥{n_min_large}")
    ax.axhline(n_min_medium, color="#ffaa00", lw=1.5, ls="--",
               label=f"Medium effect d=0.5 → n≥{n_min_medium}")
    ax.axhline(n_min_small,  color="#ff4444", lw=1.5, ls="--",
               label=f"Small effect d=0.2 → n≥{n_min_small}")
    ax.axhspan(0,            n_min_large,  alpha=0.07, color="#ff4444")
    ax.axhspan(n_min_large,  n_min_medium, alpha=0.07, color="#ffaa00")
    ax.axhspan(n_min_medium, ymax,         alpha=0.04, color="#44ff44")
    ax.set_ylim(0, ymax)
    ax.set_title(f"Group size vs required minimum N\n"
                 f"(α={alpha}, target power={int(power_target*100)}%)",
                 color="#cccccc", fontsize=11)
    ax.set_ylabel("N individuals", color="#aaaaaa")
    ax.tick_params(colors="#aaaaaa", axis="x", rotation=45)
    ax.tick_params(colors="#aaaaaa", axis="y")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")
    leg = ax.legend(fontsize=8, loc="upper right")
    leg.get_frame().set_facecolor("#111111")
    for t in leg.get_texts(): t.set_color("white")
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — Cohen's d
    # ════════════════════════════════════════════════════════
    if len(df_pairs) > 0:
        ds_vals  = df_pairs["cohens_d"].values
        d_colors = ["#44ff44" if d >= 0.8
                    else "#ffaa00" if d >= 0.5
                    else "#ff4444" for d in ds_vals]

        fig2, ax = plt.subplots(figsize=(8, max(4, len(pair_labels)*0.35)))
        fig2.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
        ax.barh(range(len(pair_labels)), ds_vals, color=d_colors,
                edgecolor="#222", linewidth=0.5, alpha=0.9)
        for i, d in enumerate(ds_vals):
            ax.text(d+0.01, i, f"{d:.2f}", va="center",
                    color="white", fontsize=7)
        ax.axvline(0.8, color="#44ff44", lw=1.5, ls="--",
                   label="Large effect (0.8)")
        ax.axvline(0.5, color="#ffaa00", lw=1.5, ls="--",
                   label="Medium effect (0.5)")
        ax.axvline(0.2, color="#ff4444", lw=1.5, ls="--",
                   label="Small effect (0.2)")
        ax.set_yticks(range(len(pair_labels)))
        ax.set_yticklabels(pair_labels, fontsize=8, color="#cccccc")
        ax.set_xlabel(f"Median Cohen's d ({feat_label})",
                      color="#aaaaaa")
        ax.set_title("Effect size between each pair of groups",
                     color="#cccccc", fontsize=11)
        ax.tick_params(colors="#aaaaaa")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        leg = ax.legend(fontsize=8, loc="lower right")
        leg.get_frame().set_facecolor("#111111")
        for t in leg.get_texts(): t.set_color("white")
        plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Power
    # ════════════════════════════════════════════════════════
    if len(df_pairs) > 0:
        powers   = df_pairs["power_obs"].values
        p_colors = ["#44ff44" if p >= power_target
                    else "#ffaa00" if p >= 0.5
                    else "#ff4444" for p in powers]

        fig3, ax = plt.subplots(figsize=(8, max(4, len(pair_labels)*0.35)))
        fig3.patch.set_facecolor("#1a1a2e"); ax.set_facecolor("#0d0d1a")
        ax.barh(range(len(pair_labels)), powers, color=p_colors,
                edgecolor="#222", linewidth=0.5, alpha=0.9)
        for i, p in enumerate(powers):
            ax.text(p+0.01, i, f"{p:.0%}", va="center",
                    color="white", fontsize=7)
        ax.axvline(power_target, color="#FFD700", lw=2, ls="--",
                   label=f"Target {int(power_target*100)}%")
        ax.axvline(0.5, color="#888888", lw=1, ls=":",
                   label="50% threshold")
        ax.set_xlim(0, 1.1)
        ax.set_yticks(range(len(pair_labels)))
        ax.set_yticklabels(pair_labels, fontsize=8, color="#cccccc")
        ax.set_xlabel("Estimated statistical power", color="#aaaaaa")
        ax.set_title(f"Power per pair — BH correction\n"
                     f"(green ≥ {int(power_target*100)}%, orange ≥ 50%, red < 50%)",
                     color="#cccccc", fontsize=11)
        ax.tick_params(colors="#aaaaaa")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        leg = ax.legend(fontsize=8)
        leg.get_frame().set_facecolor("#111111")
        for t in leg.get_texts(): t.set_color("white")
        plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 4 — Table
    # ════════════════════════════════════════════════════════
    if len(df_pairs) > 0:
        headers    = ["G1","G2","n1","n2","Cohen d",
                      "Power","p-val (BH)","Sig ?","N missing"]
        table_data = []
        for r in df_pairs.itertuples():
            n_need   = r.n_min_needed
            n_missing = (max(0, n_need - min(r.n1, r.n2))
                        if n_need < np.inf else "∞")
            table_data.append([
                r.g1, r.g2, str(r.n1), str(r.n2),
                f"{r.cohens_d:.2f}",
                f"{r.power_obs:.0%}",
                f"{r.p_value:.4f}",
                "✓" if r.significant else "✗",
                str(int(n_missing)) if isinstance(n_missing, (int,float))
                else n_missing,
            ])

        fig4, ax = plt.subplots(
            figsize=(14, max(3, len(table_data)*0.4+1)))
        fig4.patch.set_facecolor("#1a1a2e")
        ax.set_facecolor("#0d0d1a"); ax.axis("off")
        tbl = ax.table(cellText=table_data, colLabels=headers,
                       loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.6)
        for (row, col), cell in tbl.get_celld().items():
            cell.set_edgecolor("#333333")
            if row == 0:
                cell.set_facecolor("#1a1a3e")
                cell.set_text_props(color="white", fontweight="bold")
            else:
                cell.set_facecolor("#0d0d1a")
                cell.set_text_props(color="#cccccc")
                if col == 5:
                    try:
                        p = float(table_data[row-1][5].strip("%"))/100
                        cell.set_facecolor(
                            "#1a3a1a" if p >= power_target
                            else "#3a2a00" if p >= 0.5 else "#3a0000")
                    except: pass
                if col == 7:
                    cell.set_facecolor(
                        "#1a3a1a" if table_data[row-1][7]=="✓"
                        else "#3a0000")
                if col == 8:
                    try:
                        nm = int(table_data[row-1][8])
                        cell.set_facecolor(
                            "#1a3a1a" if nm == 0
                            else "#3a2a00" if nm <= 5 else "#3a0000")
                    except: pass
        ax.set_title(
            "Summary table — Benjamini-Hochberg correction\n"
            "'N missing' = additional individuals needed for the target power",
            color="#cccccc", fontsize=11, pad=15)
        plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 5 — Global score
    # ════════════════════════════════════════════════════════
    fig5, ax = plt.subplots(figsize=(7, 5))
    fig5.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#0d0d1a"); ax.axis("off")
    for sp in ax.spines.values(): sp.set_edgecolor("#333333")

    score_color = ("#44ff44" if score_global >= 0.7
                   else "#ffaa00" if score_global >= 0.4 else "#ff4444")
    verdict = ("✅ Statistically solid split"        if score_global >= 0.7
               else "🟡 Acceptable split, to be consolidated" if score_global >= 0.4
               else "🔴 Insufficient power — consider merging")

    ax.text(0.5, 0.92, f"Global score : {score_global:.0%}",
            ha="center", va="center", fontsize=24,
            color=score_color, fontweight="bold",
            transform=ax.transAxes)
    ax.text(0.5, 0.78, verdict, ha="center", va="center",
            fontsize=12, color=score_color, transform=ax.transAxes)

    for i, (label, val, col) in enumerate([
        ("Mean power between pairs (×0.5)",
         f"{score_power:.0%}",
         "#44ff44" if score_power >= 0.7 else "#ffaa00"
         if score_power >= 0.4 else "#ff4444"),
        ("Proportion of BH-significant pairs (×0.3)",
         f"{score_sig:.0%}",
         "#44ff44" if score_sig >= 0.7 else "#ffaa00"
         if score_sig >= 0.4 else "#ff4444"),
        ("Proportion of groups with OK size (×0.2)",
         f"{score_size:.0%}",
         "#44ff44" if score_size >= 0.7 else "#ffaa00"
         if score_size >= 0.4 else "#ff4444"),
    ]):
        y = 0.60 - i * 0.12
        ax.text(0.05, y, f"• {label}", ha="left", va="center",
                fontsize=9, color="#aaaaaa", transform=ax.transAxes)
        ax.text(0.95, y, val, ha="right", va="center",
                fontsize=11, color=col, fontweight="bold",
                transform=ax.transAxes)

    for i, line in enumerate([
        f"Min N large effect (d=0.8) : {n_min_large} individuals/group",
        f"Min N medium effect (d=0.5) : {n_min_medium} individuals/group",
        f"Min N small effect (d=0.2) : {n_min_small} individuals/group",
        f"Correction : Benjamini-Hochberg  |  α={alpha}",
        f"Features : {feat_label}",
    ]):
        ax.text(0.5, 0.22 - i*0.07, line, ha="center", va="center",
                fontsize=8, color="#666666", transform=ax.transAxes,
                fontfamily="monospace")

    ax.set_title(f"Summary — {n_tot} groups | α={alpha} | "
                 f"target power={int(power_target*100)}%",
                 color="#cccccc", fontsize=11)
    plt.tight_layout(); plt.show()

    print(f"\n{'='*55}")
    print(f"POWER — {n_tot} groups | α={alpha} | "
          f"target={int(power_target*100)}%")
    print(f"  Features  : {feat_label}")
    print(f"  Correction: Benjamini-Hochberg (FDR)")
    print(f"  Min N (d=0.8): {n_min_large} | (d=0.5): {n_min_medium} "
          f"| (d=0.2): {n_min_small}")
    print(f"  Sizes     : { {g: n for g, n in group_sizes.items()} }")
    print(f"  Score     : {score_global:.0%}  ({verdict})")

    return df_pairs




# ══════════════════════════════════════════════════════════════

def optimal_regrouping(
    final_state,
    df,
    alpha         = 0.05,
    power_target  = 0.80,
    target_effect = 0.5,
    min_groups    = 2,
    d_max_fusion  = 0.8,
    verbose       = True,
):
    """
    Iteratively merges the groups using the SeqVec D matrix
    — consistent with the initial clustering.

    d_max_fusion : maximum Cohen's d allowed for a forced merge.
                   If the closest pair has d > d_max_fusion,
                   we stop even if some groups are too small.
                   Avoids merging very different groups.
    """
    seqs = final_state["seqs"]
    D    = final_state["D"]
    ids  = final_state["multi"]   # list in the same order as D

    # Minimum N for the target effect
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta  = stats.norm.ppf(power_target)
    n_min   = int(np.ceil((z_alpha + z_beta)**2 / (2 * target_effect**2)))

    # Current mapping
    mapping = {str(aid): g for aid, g in final_state["strat"].items()}
    for m in final_state["mono"]:
        mapping[str(m)] = "MONO"

    # Index of the individuals in the D matrix
    id_to_idx = {str(aid): i for i, aid in enumerate(ids)}

    history = []

    def get_sizes(mapping):
        return dict(Counter(v for v in mapping.values() if v != "MONO"))

    def get_distance_D(g1, g2):
        """
        Mean distance in SeqVec space between the two groups.
        Uses the already computed D matrix → consistent with the clustering.
        """
        idx1 = [id_to_idx[aid] for aid in mapping
                if mapping[aid] == g1 and aid in id_to_idx]
        idx2 = [id_to_idx[aid] for aid in mapping
                if mapping[aid] == g2 and aid in id_to_idx]
        if not idx1 or not idx2: return np.inf
        sub = D[np.ix_(idx1, idx2)]
        return float(sub.mean())

    def all_ok(mapping, n_min, min_groups):
        sizes = get_sizes(mapping)
        if len(sizes) <= min_groups: return True
        return all(n >= n_min for n in sizes.values())

    if verbose:
        sizes_init = get_sizes(mapping)
        print(f"\n{'='*60}")
        print(f"OPTIMAL REGROUPING")
        print(f"  features     : SeqVec (D matrix of the clustering)")
        print(f"  n_min (d={target_effect}, power={int(power_target*100)}%) = {n_min}")
        print(f"  d_max_fusion : {d_max_fusion}")
        print(f"  Initial groups : {sizes_init}")
        print(f"{'='*60}")

        # Also show the recommended target_effect values
        if len(final_state.get("D", [])) > 0:
            df_pairs_preview = []
            non_mono = sorted(set(v for v in mapping.values() if v != "MONO"))
            for g1, g2 in combinations(non_mono, 2):
                d = get_distance_D(g1, g2)
                df_pairs_preview.append(d)
            if df_pairs_preview:
                print(f"  target_effect_auto   : "
                      f"{np.mean(df_pairs_preview):.3f}")
                print(f"  target_effect_median : "
                      f"{np.median(df_pairs_preview):.3f}")
                print(f"  target_effect_p25    : "
                      f"{np.percentile(df_pairs_preview, 25):.3f}")

    iteration = 0
    while not all_ok(mapping, n_min, min_groups):
        sizes    = get_sizes(mapping)
        non_mono = sorted(sizes.keys())

        if len(non_mono) <= min_groups:
            if verbose: print(f"[STOP] Minimum number of groups reached")
            break

        small_groups = {g for g in non_mono if sizes[g] < n_min}
        best_pair, best_score, best_d = None, np.inf, np.inf

        for g1, g2 in combinations(non_mono, 2):
            d            = get_distance_D(g1, g2)
            involves_sm  = g1 in small_groups or g2 in small_groups
            score        = d if involves_sm else d + 100
            if score < best_score:
                best_score = score
                best_pair  = (g1, g2)
                best_d     = d

        if best_pair is None: break

        # Check whether the merge is reasonable
        if best_d > d_max_fusion and not (best_pair[0] in small_groups
                                           or best_pair[1] in small_groups):
            if verbose:
                print(f"[STOP] Next merge too different "
                      f"(d={best_d:.3f} > {d_max_fusion})")
            break

        g1, g2    = best_pair
        reason    = ("insufficient size" if g1 in small_groups
                     or g2 in small_groups else "most similar groups")
        # New label = smallest label (alphabetical)
        # avoids always absorbing into the large group
        new_label = min(g1, g2)
        mapping   = {aid: (new_label if g in (g1, g2) else g)
                     for aid, g in mapping.items()}
        iteration += 1

        record = {
            "iteration": iteration, "g1": g1, "g2": g2,
            "new_label": new_label, "distance_D": round(best_d, 3),
            "n_g1": sizes[g1], "n_g2": sizes[g2],
            "n_merged": sizes[g1] + sizes[g2], "reason": reason,
        }
        history.append(record)

        if verbose:
            new_sizes = get_sizes(mapping)
            print(f"  [{iteration}] {g1}(n={sizes[g1]}) + "
                  f"{g2}(n={sizes[g2]}) → {new_label}"
                  f"(n={sizes[g1]+sizes[g2]})  "
                  f"d={best_d:.3f}  [{reason}]")
            print(f"       Remaining : {new_sizes}")

    if verbose:
        final_sizes = get_sizes(mapping)
        n_ok = sum(1 for n in final_sizes.values() if n >= n_min)
        print(f"\n[RESULT] {len(final_sizes)} groups | "
              f"{n_ok}/{len(final_sizes)} reach n≥{n_min}")
        print(f"  Final sizes : {final_sizes}")

    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Before / After
    # ════════════════════════════════════════════════════════
    sizes_before = get_sizes({str(aid): g
                              for aid, g in final_state["strat"].items()})
    sizes_after = get_sizes(mapping)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor("#1a1a2e")

    for ax, (sizes, title) in zip(axes, [
        (sizes_before, f"Before  ({len(sizes_before)} groups)"),
        (sizes_after, f"After  ({len(sizes_after)} groups)"),
    ]):
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.tick_params(colors="#aaaaaa")
        groups = sorted(sizes.keys())
        vals   = [sizes[g] for g in groups]
        colors = [STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                  for i in range(len(groups))]
        bars = ax.bar(groups, vals, color=colors,
                      edgecolor="#222", linewidth=0.8, alpha=0.9)
        for bar, n in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, n+0.05, str(n),
                    ha="center", va="bottom", color="white",
                    fontsize=9, fontweight="bold")
        ax.axhline(n_min, color="#FFD700", lw=2, ls="--",
                   label=f"n_min={n_min} (d={target_effect})")
        ax.axhspan(0, n_min, alpha=0.06, color="#ff4444")
        ax.axhspan(n_min, max(vals+[n_min])*1.3, alpha=0.04, color="#44ff44")
        ax.set_ylim(0, max(vals+[n_min])*1.3)
        ax.set_title(title, color="#cccccc", fontsize=11)
        ax.set_ylabel("N individuals", color="#aaaaaa")
        ax.tick_params(axis="x", rotation=45)
        leg = ax.legend(fontsize=8)
        leg.get_frame().set_facecolor("#111111")
        for t in leg.get_texts(): t.set_color("white")

    fig.suptitle(
        f"Optimal regrouping | n_min={n_min} | "
        f"{len(history)} merges | SeqVec distance",
        color="#cccccc", fontsize=12)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — Merge log
    # ════════════════════════════════════════════════════════
    if history:
        df_hist = pd.DataFrame(history)
        fig2, axes2 = plt.subplots(1, 2,
                                    figsize=(14, max(3, len(history)*0.5+1)))
        fig2.patch.set_facecolor("#1a1a2e")

        labels_fus = [f"{r.g1}+{r.g2}→{r.new_label}"
                      for r in df_hist.itertuples()]

        # SeqVec distance of the merges
        ax = axes2[0]
        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        d_colors = ["#44ff44" if d < 2 else "#ffaa00" if d < 5 else "#ff4444"
                    for d in df_hist["distance_D"]]
        ax.barh(range(len(labels_fus)), df_hist["distance_D"],
                color=d_colors, edgecolor="#222", alpha=0.9)
        for i, (d, r) in enumerate(zip(df_hist["distance_D"],
                                        df_hist.itertuples())):
            ax.text(d+0.05, i, f"d={d:.2f}  [{r.reason}]",
                    va="center", color="#cccccc", fontsize=8)
        ax.set_yticks(range(len(labels_fus)))
        ax.set_yticklabels(labels_fus, fontsize=8, color="#cccccc")
        ax.set_xlabel("Mean SeqVec distance between the groups",
                      color="#aaaaaa")
        ax.set_title("Order and reason of the merges\n"
                     "(green=very close, red=forced by size)",
                     color="#cccccc", fontsize=10)
        ax.tick_params(colors="#aaaaaa")

        # Merged sizes
        ax2 = axes2[1]
        ax2.set_facecolor("#0d0d1a")
        for sp in ax2.spines.values(): sp.set_edgecolor("#333333")
        ax2.barh(range(len(df_hist)), df_hist["n_g1"],
                 color="#4466aa", alpha=0.8, label="n(G1)", edgecolor="#222")
        ax2.barh(range(len(df_hist)), df_hist["n_g2"],
                 left=df_hist["n_g1"],
                 color="#aa6644", alpha=0.8, label="n(G2)", edgecolor="#222")
        ax2.axvline(n_min, color="#FFD700", lw=2, ls="--",
                    label=f"n_min={n_min}")
        ax2.set_yticks(range(len(labels_fus)))
        ax2.set_yticklabels(labels_fus, fontsize=8, color="#cccccc")
        ax2.set_xlabel("Size of the merged groups", color="#aaaaaa")
        ax2.set_title("Size of the groups before merging",
                      color="#cccccc", fontsize=10)
        ax2.tick_params(colors="#aaaaaa")
        leg2 = ax2.legend(fontsize=8)
        leg2.get_frame().set_facecolor("#111111")
        for t in leg2.get_texts(): t.set_color("white")

        fig2.suptitle("Merge details", color="#cccccc", fontsize=11)
        plt.tight_layout(); plt.show()

    non_mono_groups = sorted(set(v for v in mapping.values() if v != "MONO"))
    num_to_letter   = {g: chr(65+i) for i, g in enumerate(non_mono_groups)}

    mapping_letters = {
        aid: (num_to_letter[g] if g != "MONO" else "MONO")
        for aid, g in mapping.items()
    }

    print(f"\n[LETTERS] {len(non_mono_groups)} groups → "
          f"{list(num_to_letter.values())}")

    return mapping_letters, history 

In [12]:
# Depending on what you want to detect:
#target_effect = 0.8   # large effect  → n_min ≈ 26  → few merges
#target_effect = 0.5   # medium effect → n_min ≈ 64  → moderate merges
#target_effect = 0.2   # small effect  → n_min ≈ 394 → many merges

# For your data (200 individuals, ~20 initial groups)
# → 0.5 is a reasonable starting point
mapping_optimal, history = optimal_regrouping(
    final_state,
    df,
    alpha         = 0.05,
    power_target  = 0.95,
    target_effect = 0.75,   # ← fixed
    min_groups    = 2,
    d_max_fusion  = 0.8,
    verbose       = True,
)


OPTIMAL REGROUPING
  features     : SeqVec (D matrix of the clustering)
  n_min (d=0.75, power=95%) = 12
  d_max_fusion : 0.8
  Initial groups : {'0': 44, '1': 32, '3': 27, '5': 23, '4': 26, '2': 31}
  target_effect_auto   : 2.899
  target_effect_median : 2.910
  target_effect_p25    : 2.783

[RESULT] 6 groups | 6/6 reach n≥12
  Final sizes : {'0': 44, '1': 32, '3': 27, '5': 23, '4': 26, '2': 31}

[LETTERS] 6 groups → ['A', 'B', 'C', 'D', 'E', 'F']


In [13]:
# ── Create final_state_optimal with the LETTERS of mapping_optimal ──
# ── Create final_state_optimal with the LETTERS of mapping_optimal ──
final_state_optimal = dict(final_state)

final_state_optimal["strat"] = {
    aid: g
    for aid, g in mapping_optimal.items()
    if g != "MONO"
}
final_state_optimal["mono"] = [
    aid for aid, g in mapping_optimal.items()
    if g == "MONO"
]

# ── ADD mapping_letters ────────────────────────────────────
# mapping_optimal already contains letters (A, B, C...)
# so mapping_letters = mapping_optimal directly
final_state_optimal["mapping_letters"] = dict(mapping_optimal)

# Check
print("strat groups     :", sorted(set(final_state_optimal["strat"].values())))
print("mapping_letters  :", sorted(set(final_state_optimal["mapping_letters"].values())))
print("n_individuals    :", len(final_state_optimal["strat"]))
# ── Calls ────────────────────────────────────────────────────

# Option A — raw features (recommended for the power analysis)
df_pairs = show_power_analysis(
    final_state_optimal, df,
    features         = FEATURES,
    use_raw_features = True,
    top_pct_features = 1.0,
    alpha            = 0.05,
    power_target     = 0.80,
)

# # Option B — SeqVec top 20% discriminant features
# df_pairs = show_power_analysis(
#     final_state_optimal, df,
#     use_raw_features = False,
#     top_pct_features = 0.20,
#     alpha            = 0.05,
#     power_target     = 0.80,
# )

strat groups     : ['A', 'B', 'C', 'D', 'E', 'F']
mapping_letters  : ['A', 'B', 'C', 'D', 'E', 'F', 'MONO']
n_individuals    : 183
[BH] 15/15 significant pairs after Benjamini-Hochberg correction

POWER — 6 groups | α=0.05 | target=80%
  Features  : raw features (6)
  Correction: Benjamini-Hochberg (FDR)
  Min N (d=0.8): 7 | (d=0.5): 16 | (d=0.2): 99
  Sizes     : {'A': 44, 'B': 32, 'C': 31, 'D': 27, 'E': 26, 'F': 23, 'MONO': 17}
  Score     : 97%  (✅ Statistically solid split)


C:\Users\nonoh\AppData\Local\Temp\ipykernel_10444\4107449365.py:503: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout(); plt.show()


## Strategy profiles (medoids, letters) — computed here, displayed in 2_model

In [14]:
#def plot_strategy_profiles

from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors


def plot_strategy_profiles(mapping_optimal, final_state, df, features=FEATURES,
                           cmap_cl=None, norm_cl=None):

    # Default values if not provided
    n_cl_local = final_state["n_cl"]
    if cmap_cl is None:
        cmap_cl = CMAP_CLUSTER
    if norm_cl is None:
        norm_cl = NORM_CLUSTER

    seqs = final_state["seqs"]
    n_cl = final_state["n_cl"]

    df_w = df.copy()
    df_w["animal_id"] = df_w["animal_id"].astype(str)

    unique_groups = sorted(set(v for v in mapping_optimal.values() if v != "MONO"))
    num_to_letter = {g: chr(65+i) for i, g in enumerate(unique_groups)}
    num_to_letter["MONO"] = "MONO"
    mapping_letters  = {aid: num_to_letter[g] for aid, g in mapping_optimal.items()}
    strategy_letters = sorted(set(v for v in mapping_letters.values() if v != "MONO"))
    n_strat          = len(strategy_letters)

    pal         = {s: STRATEGY_PALETTE[i % len(STRATEGY_PALETTE)]
                   for i, s in enumerate(strategy_letters)}
    pal["MONO"] = "#666666"

    # ── DTW for the medoid ──────────────────────────────────────
    def _dtw(seq_a, seq_b, n_cl):
        if not seq_a or not seq_b: return np.inf
        centers = np.eye(n_cl)
        tA = np.array([centers[min(c, n_cl-1)] for c in seq_a])
        tB = np.array([centers[min(c, n_cl-1)] for c in seq_b])
        TA, TB = len(tA), len(tB)
        cost   = cdist(tA, tB, metric='euclidean')
        D      = np.full((TA, TB), np.inf)
        D[0,0] = cost[0,0]
        for i in range(1, TA): D[i,0] = D[i-1,0] + cost[i,0]
        for j in range(1, TB): D[0,j] = D[0,j-1] + cost[0,j]
        for i in range(1, TA):
            for j in range(1, TB):
                D[i,j] = cost[i,j] + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
        return D[TA-1, TB-1]

    def _medoid(members):
        n = len(members)
        if n == 1: return members[0], seqs[members[0]]["seq"]
        dists = np.zeros(n)
        for i in range(n):
            for j in range(n):
                if i != j:
                    dists[i] += _dtw(seqs[members[i]]["seq"],
                                     seqs[members[j]]["seq"], n_cl)
        best = int(np.argmin(dists))
        return members[best], seqs[members[best]]["seq"]

    angles   = np.linspace(0, 2*np.pi, n_cl, endpoint=False)
    ang_full = np.concatenate([angles, [angles[0]]])

    profiles = {}
    for s in strategy_letters:
        members         = [aid for aid, g in mapping_letters.items() if g == s]
        members_in_seqs = [m for m in members if m in seqs]
        if not members_in_seqs: continue

        hists = []
        for aid in members_in_seqs:
            sq, lg = seqs[aid]["seq"], seqs[aid]["lengths"]
            total  = sum(lg)
            if total == 0: continue
            h = np.zeros(n_cl)
            for c, l in zip(sq, lg):
                if c < n_cl: h[c] += l / total
            hists.append(h)
        mean_hist = np.mean(hists, axis=0) if hists else np.zeros(n_cl)

        print(f"  [MEDOID] Strategy {s} ({len(members_in_seqs)} ind)...", end=" ")
        medoid_aid, modal_compressed = _medoid(members_in_seqs)
        print(f"-> ID {medoid_aid}")

        durations = {c: [] for c in range(n_cl)}
        for aid in members_in_seqs:
            for c, l in zip(seqs[aid]["seq"], seqs[aid]["lengths"]):
                if c < n_cl: durations[c].append(l)
        median_durations = {c: np.median(durations[c]) if durations[c] else 0
                         for c in range(n_cl)}

        trans_count = {}
        for aid in members_in_seqs:
            sq = seqs[aid]["seq"]
            for a, b in zip(sq[:-1], sq[1:]):
                trans_count[(a,b)] = trans_count.get((a,b), 0) + 1
        top_trans = sorted(trans_count.items(), key=lambda x: -x[1])[:3]

        feat_means = {}
        if features:
            ids_str = [str(m) for m in members_in_seqs]
            mask    = df_w["animal_id"].isin(ids_str)
            if mask.any():
                feat_means = df_w[mask][features].mean().to_dict()

        profiles[s] = {
            "members"         : members_in_seqs,
            "all_members"     : members,
            "n"               : len(members_in_seqs),
            "mean_hist"       : mean_hist,
            "modal_seq"       : modal_compressed,
            "medoid_aid"      : medoid_aid,
            "median_durations"   : median_durations,
            "top_transitions" : top_trans,
            "dominant_cluster": int(np.argmax(mean_hist)),
            "feat_means"      : feat_means,
            "mean_seq_len"    : np.mean([len(seqs[a]["seq"])
                                         for a in members_in_seqs]),
            "mean_n_clusters" : np.mean([len(set(seqs[a]["seq"]))
                                         for a in members_in_seqs]),
        }
    print(f"[DEBUG] mapping_optimal sample : {list(mapping_optimal.items())[:5]}")
    print(f"[DEBUG] mapping_letters sample : {list(mapping_letters.items())[:5]}")
    print(f"[DEBUG] strategy_letters : {strategy_letters}")
    print(f"[DEBUG] seqs keys sample : {list(seqs.keys())[:5]}")

    for s in strategy_letters:
        members = [aid for aid, g in mapping_letters.items() if g == s]
        in_seqs = [m for m in members if m in seqs]
        print(f"  {s} : {len(members)} members, {len(in_seqs)} in seqs")
    # ════════════════════════════════════════════════════════
    # FIGURE 1 — Radars
    # ════════════════════════════════════════════════════════
    n_cols = min(4, n_strat)
    n_rows = (n_strat + n_cols - 1) // n_cols
    fig1   = plt.figure(figsize=(n_cols * 5, n_rows * 5))
    fig1.patch.set_facecolor("#1a1a2e")
    fig1.suptitle("Behavioural profiles of the strategies",
                  color="#cccccc", fontsize=13, y=1.01)

    for si, s in enumerate(strategy_letters):
        if s not in profiles: continue
        p   = profiles[s]
        col = pal[s]
        ax  = fig1.add_subplot(n_rows, n_cols, si+1, polar=True)
        ax.set_facecolor("#0d0d1a")
        vals_f = np.concatenate([p["mean_hist"], [p["mean_hist"][0]]])
        ax.plot(ang_full, vals_f, lw=2.5, color=col)
        ax.fill(ang_full, vals_f, alpha=0.3, color=col)
        ax.set_xticks(angles)
        ax.set_xticklabels([f"C{c}" for c in range(n_cl)],
                            fontsize=7, color="#aaaaaa")
        ax.set_ylim(0, max(p["mean_hist"].max()*1.2, 0.01))
        ax.set_yticks([])
        ax.grid(color="#333333", lw=0.5)
        ax.spines["polar"].set_color("#333333")
        modal_str = "→".join([f"C{c}" for c in p["modal_seq"][:6]])
        if len(p["modal_seq"]) > 6: modal_str += "…"
        ax.set_title(
            f"Strategy {s}  (n={p['n']})\n"
            f"Dominant : C{p['dominant_cluster']}\n"
            f"Medoid : ID {p['medoid_aid']}\n"
            f"{modal_str}",
            color=col, fontsize=9, pad=12
        )
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 2 — MEDOID sequence (1 real individual)
    # Visual improvement: merged blocks, thickness by duration
    # ════════════════════════════════════════════════════════
    fig2, axes = plt.subplots(n_strat, 1,
                               figsize=(14, max(3, n_strat * 1.6)))
    fig2.patch.set_facecolor("#1a1a2e")
    fig2.suptitle(
        "Medoid sequence per strategy\n"
        "(most representative real individual — DTW medoid)\n"
        "width ∝ duration | consecutive identical blocks merged",
        color="#cccccc", fontsize=11
    )
    if n_strat == 1: axes = [axes]

    for si, (s, ax) in enumerate(zip(strategy_letters, axes)):
        if s not in profiles: continue
        p    = profiles[s]
        col  = pal[s]

        # Sequence of the medoid (real individual)
        seq_raw  = seqs[p["medoid_aid"]]["seq"]
        lgts_raw = seqs[p["medoid_aid"]]["lengths"]

        # ── Merge consecutive identical blocks ───────────
        # Avoids the visual repetition C0 C0 C0 → a single large C0
        seq_fused, lgts_fused = [], []
        for c, l in zip(seq_raw, lgts_raw):
            if seq_fused and seq_fused[-1] == c:
                lgts_fused[-1] += l
            else:
                seq_fused.append(c)
                lgts_fused.append(l)

        ax.set_facecolor("#0d0d1a")
        for sp in ax.spines.values(): sp.set_edgecolor("#333333")
        ax.set_yticks([])
        ax.tick_params(colors="#555555", labelsize=7)

        x    = 0
        dmax = max(lgts_fused) if lgts_fused else 1
        for c, dur in zip(seq_fused, lgts_fused):
            c_color = mcolors.to_hex(cmap_cl(norm_cl(c)))

            # Height proportional to duration → important blocks = thicker
            h = 0.35 + 0.55 * (dur / dmax)

            ax.barh(0, dur, left=x, height=h,
                    color=c_color, edgecolor="#0d0d1a", lw=0.8,
                    alpha=0.9)

            # Label only if the block is wide enough
            if dur >= dmax * 0.04:
                fontsize = min(9, max(5, int(7 * dur / dmax * 3)))
                ax.text(x + dur/2, 0, f"C{c}",
                        ha="center", va="center",
                        fontsize=fontsize, color="white",
                        fontweight="bold")
            x += dur

        ax.set_xlim(0, x)
        ax.set_ylim(-0.55, 0.55)
        # shrimp sessions: start of the cuttlefish activity for this individual
        _ph = phase_change(df)
        if _ph and int(p["medoid_aid"]) in _ph:
            mark_phase(ax, _ph[int(p["medoid_aid"])] * x, label=si == 0)

        # Group colour bar on the left
        ax.barh(0, x * 0.008, left=-x*0.015, height=1.1,
                color=col, alpha=0.9)

        total_frames = sum(lgts_fused)
        n_blocks_orig = len(seq_raw)
        n_blocks_fused  = len(seq_fused)

        trans_str = "  |  ".join(
            [f"C{a}→C{b} ({n}×)" for (a,b), n in p["top_transitions"]]
        )

        ax.set_ylabel(
            f"Strat. {s}\nn={p['n']}\nID {p['medoid_aid']}",
            color=col, fontsize=8, rotation=0,
            ha="right", va="center", labelpad=60
        )
        ax.text(x * 1.005, 0.25,
                f"{total_frames}f",
                va="center", ha="left", fontsize=6, color="#666666")
        ax.text(x * 1.005, -0.25,
                f"{n_blocks_fused} blocks ({n_blocks_orig} raw)",
                va="center", ha="left", fontsize=6, color="#555555")
        ax.text(x * 1.005, 0.0,
                f"↑ {trans_str}",
                va="center", ha="left", fontsize=6, color="#777777")

    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 3 — Summary table + list of individuals
    # ════════════════════════════════════════════════════════
    fig3, ax = plt.subplots(figsize=(18, max(3, n_strat * 0.9 + 1.5)))
    fig3.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#0d0d1a"); ax.axis("off")

    headers = (["Strategy", "N", "Medoid", "Dominant cluster",
                "Mean blocks", "Distinct clusters", "Top transition",
                "Individuals (IDs)"]
               + [f.replace("_"," ")[:10] for f in features])

    table_data = []
    for s in strategy_letters:
        if s not in profiles: continue
        p = profiles[s]

        top_t = (f"C{p['top_transitions'][0][0][0]}→"
                 f"C{p['top_transitions'][0][0][1]}"
                 if p["top_transitions"] else "—")

        # List of individuals — sorted, comma-separated
        ids_str = ", ".join(sorted([str(m) for m in p["members"]]))
        # Truncate if too long for the table
        if len(ids_str) > 60:
            ids_str = ids_str[:57] + "…"

        row = [
            f"  {s}  ",
            str(p["n"]),
            str(p["medoid_aid"]),
            f"C{p['dominant_cluster']}  ({p['mean_hist'][p['dominant_cluster']]:.0%})",
            f"{p['mean_seq_len']:.1f}",
            f"{p['mean_n_clusters']:.1f}",
            top_t,
            ids_str,
        ]
        for feat in features:
            v = p["feat_means"].get(feat, np.nan)
            row.append(f"{v:.2f}" if not np.isnan(v) else "—")
        table_data.append(row)

    tbl = ax.table(cellText=table_data, colLabels=headers,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 2.2)

    for (row, col_idx), cell in tbl.get_celld().items():
        cell.set_edgecolor("#333333")
        if row == 0:
            cell.set_facecolor("#1a1a3e")
            cell.set_text_props(color="white", fontweight="bold")
        else:
            s = strategy_letters[row-1] if row-1 < len(strategy_letters) else None
            cell.set_facecolor("#0d0d1a")
            cell.set_text_props(color="#cccccc")
            if col_idx == 0 and s:
                cell.set_facecolor(pal.get(s, "#0d0d1a") + "55")
                cell.set_text_props(color=pal.get(s, "white"),
                                    fontweight="bold")
            # Individuals column: lightly coloured
            if col_idx == 7 and s:
                cell.set_facecolor(pal.get(s, "#0d0d1a") + "22")
                cell.set_text_props(color="#aaaaaa", fontsize=6)

    ax.set_title("Summary table — DTW medoid + list of individuals",
                 color="#cccccc", fontsize=11, pad=15)
    plt.tight_layout(); plt.show()

    # ════════════════════════════════════════════════════════
    # FIGURE 4 — Detailed list of individuals per group
    # ════════════════════════════════════════════════════════
    fig4, ax4 = plt.subplots(figsize=(14, max(4, n_strat * 1.2 + 1)))
    fig4.patch.set_facecolor("#1a1a2e")
    ax4.set_facecolor("#0d0d1a"); ax4.axis("off")

    ax4.set_title("Individuals per strategy",
                  color="#cccccc", fontsize=12, pad=10)

    y_pos = 0.97
    step  = 1.0 / (n_strat + 1)

    for s in strategy_letters:
        if s not in profiles: continue
        p   = profiles[s]
        col = pal[s]

        ids_sorted = sorted([str(m) for m in p["members"]])
        ids_chunks = [ids_sorted[i:i+20] for i in range(0, len(ids_sorted), 20)]

        # Group label
        ax4.text(0.01, y_pos,
                 f"Strategy {s}  (n={p['n']})  —  medoid : {p['medoid_aid']}",
                 transform=ax4.transAxes,
                 color=col, fontsize=10, fontweight='bold', va='top')

        # IDs on one or more lines
        for li, chunk in enumerate(ids_chunks):
            ax4.text(0.12, y_pos - 0.015 * (li + 1),
                     "  ".join(chunk),
                     transform=ax4.transAxes,
                     color="#cccccc", fontsize=7, va='top',
                     fontfamily='monospace')

        y_pos -= step

    plt.tight_layout(); plt.show()

    # ── Terminal summary ───────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"PROFILES ({n_strat} groups) — DTW medoids")
    print(f"{'='*60}")
    for s in strategy_letters:
        if s not in profiles: continue
        p = profiles[s]
        ids_sorted = sorted([str(m) for m in p["members"]])
        print(f"\nStrategy {s}  (n={p['n']})  —  medoid : ID {p['medoid_aid']}")
        print(f"  Individuals : {', '.join(ids_sorted)}")
        print(f"  Dominant  : C{p['dominant_cluster']} "
              f"({p['mean_hist'][p['dominant_cluster']]:.0%})")
        print(f"  Mean blocks : {p['mean_seq_len']:.1f} | "
              f"Distinct clusters : {p['mean_n_clusters']:.1f}")
        if p["top_transitions"]:
            print(f"  Top trans : " +
                  ", ".join([f"C{a}→C{b}({n}×)"
                             for (a,b),n in p["top_transitions"]]))

    return profiles, mapping_letters

In [15]:
# Cell 19 — must be run BEFORE cell 30
profiles, mapping_letters = plot_strategy_profiles(
    mapping_optimal,
    final_state,
    df,
    features = FEATURES,
)

  [MEDOID] Strategy A (44 ind)... -> ID 194
  [MEDOID] Strategy B (32 ind)... -> ID 112
  [MEDOID] Strategy C (31 ind)... -> ID 3
  [MEDOID] Strategy D (27 ind)... -> ID 83
  [MEDOID] Strategy E (26 ind)... -> ID 156
  [MEDOID] Strategy F (23 ind)... -> ID 122
[DEBUG] mapping_optimal sample : [('1', 'A'), ('100', 'B'), ('101', 'D'), ('102', 'F'), ('103', 'E')]
[DEBUG] mapping_letters sample : [('1', 'A'), ('100', 'B'), ('101', 'D'), ('102', 'F'), ('103', 'E')]
[DEBUG] strategy_letters : ['A', 'B', 'C', 'D', 'E', 'F']
[DEBUG] seqs keys sample : ['1', '10', '100', '101', '102']
  A : 44 members, 44 in seqs
  B : 32 members, 32 in seqs
  C : 31 members, 31 in seqs
  D : 27 members, 27 in seqs
  E : 26 members, 26 in seqs
  F : 23 members, 23 in seqs

PROFILES (6 groups) — DTW medoids

Strategy A  (n=44)  —  medoid : ID 194
  Individuals : 1, 104, 111, 124, 127, 134, 135, 140, 143, 168, 170, 175, 182, 183, 187, 193, 194, 195, 197, 20, 25, 31, 36, 37, 41, 43, 44, 46, 47, 51, 56, 57, 59, 6, 

## Cluster statistics

In [16]:
#  Additional imports for trajectory analysis
import ast
from scipy.fft import fft
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

# FEATURES and PALETTE already defined in cell 0
N_CLUSTERS = final_state["n_cl"]
print(f" Imports OK | N_CLUSTERS={N_CLUSTERS} | FEATURES={FEATURES}")

 Imports OK | N_CLUSTERS=23 | FEATURES=['distance_to_shrimp', 'attack_side', 'bearing_to_prey', 'cuttle_approach_speed', 'orbit_speed', 'who_moves']


In [17]:
#  Cluster centres in feature space
# Computed directly from df (already loaded in memory)
cluster_col = final_state["col"]   # "supercluster" or "cluster"
n_cl        = final_state["n_cl"]

cluster_ids        = sorted(df[cluster_col].unique())
cluster_stats_rows = []
for k in cluster_ids:
    mask = df[cluster_col] == k
    cluster_stats_rows.append(df.loc[mask, FEATURES].mean().values)

cluster_stats = pd.DataFrame(cluster_stats_rows, columns=FEATURES,
                              index=cluster_ids)
final_state["cluster_stats"] = cluster_stats

seqs_dict = final_state["seqs"]
n_mono    = sum(1 for s in seqs_dict.values() if len(s["seq"]) <= 1)

print(f"✅ cluster_stats : {cluster_stats.shape}")
print(cluster_stats.round(4))
print(f"\n✅ {len(seqs_dict)} individuals including {n_mono} MONO")
# mapping_letters is already defined (returned by plot_strategy_profiles)
print(f"✅ mapping_letters : {len(mapping_letters)} entries")

✅ cluster_stats : (23, 6)
    distance_to_shrimp  attack_side  bearing_to_prey  cuttle_approach_speed  \
0               5.6052     119.9527           6.9268                 2.9954   
1               5.6343      40.9720          11.9980                 2.8457   
2               5.8095      58.5273           9.0606                -0.3033   
3               5.9827      37.8268          28.0853                 0.0323   
4               5.4299     109.4555          25.7034                 0.0914   
5               5.3521     136.2606           8.9341                 3.0214   
6               4.9709      81.8770          20.1204                -1.9142   
7               5.1856     135.2914           8.6192                 2.8138   
8               6.3581      70.0903          35.6678                 0.0059   
9               5.6322     147.1279          13.3861                 0.2262   
10              5.3897     101.2345          18.4255                 0.7228   
11              5.6388    

## Save / reload a session

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SESSION SAVE — into a NAMED folder
#  Everything needed to resume or compare: configuration, clusters,
#  UMAP, sequences, SeqVec distance, groups (explorer + regrouping),
#  group explanation, figures and a copy of the code. Folder: OUT_DIR/session_save/<SAVE_NAME>
# ══════════════════════════════════════════════════════════════
DO_SAVE   = True            # ⚠ SAFETY: set True then rerun THIS cell to save
SAVE_NAME = "Test_superCluster"          # folder name, e.g. "before_duration_tests"
SAVE_NOTE = ""                   # free note (what this session contains)
OVERWRITE = False                # True = replaces an existing folder with the same name

if not DO_SAVE:
    print("[SAVE] disabled (DO_SAVE = False): nothing was written. "
          "To save: DO_SAVE = True, then rerun this cell.")
else:

    import io, json, pickle, shutil, contextlib, traceback
    from datetime import datetime
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    SAVE_DIR = RESULTS_DIR / SAVE_NAME
    if SAVE_DIR.exists() and any(SAVE_DIR.iterdir()) and not OVERWRITE:
        raise FileExistsError(f"This folder already exists (change SAVE_NAME or set OVERWRITE = True): {SAVE_DIR}")
    if SAVE_DIR.exists() and OVERWRITE:
        shutil.rmtree(SAVE_DIR)
    FIG_DIR = SAVE_DIR / "figures"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[SAVE] → {SAVE_DIR}\n")
    _report = []
    # figures created WITHOUT a window (no backend switch, no Qt popup)
    _was_interactive = plt.isinteractive()
    plt.ioff()


    def _g(name):
        return globals().get(name)


    def _save(label, fn):
        try:
            out = fn()
            if out is False:
                _report.append(("SKIP", label)); print(f"  [SKIP] {label}")
            else:
                _report.append(("OK", label)); print(f"  [OK]   {label}")
        except Exception as e:
            _report.append(("ERR", label)); print(f"  [ERR]  {label}  ({type(e).__name__}: {e})")


    def _jsonable(o):
        if isinstance(o, dict):
            return {str(k): _jsonable(v) for k, v in o.items()}
        if isinstance(o, (list, tuple, set)):
            return [_jsonable(v) for v in o]
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, Path):
            return str(o)
        return o


    # ── 1. Configuration (cell 0 + explorer settings) ──
    def _cfg():
        fs = _g("final_state") or {}
        cfg = {
            "name": SAVE_NAME, "note": SAVE_NOTE, "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "DATASET": _g("DATASET"), "SMOOTH_FILES": _g("SMOOTH_FILES"), "OUT_DIR": OUT_DIR,
            "SELECTED_WINDOW": _g("SELECTED_WINDOW"), "FEATURES": _g("FEATURES"),
            "FEATURE_WEIGHTS": _g("FEATURE_WEIGHTS"), "EXCLUDE_ANIMALS": _g("EXCLUDE_ANIMALS"),
            "SUBJECT": _g("SUBJECT") or "cuttle", "SPEED_COL": _g("SPEED_COL"),
            "ORBIT_MIN_DEG_S": _g("ORBIT_MIN_DEG_S"), "FRAME_STEP": _g("FRAME_STEP"),
            "LEIDEN_K": _g("LEIDEN_K"), "LEIDEN_RES": _g("LEIDEN_RES"),
            "LEIDEN_K2": _g("LEIDEN_K2"), "LEIDEN_RES2": _g("LEIDEN_RES2"),
            "explorer": {k: fs.get(k) for k in ("col", "n_s", "n_cl", "seqvec_weights", "knn_k",
                                                 "min_run_len", "desc")},
            "n_groups_explorer": len({g for g in fs.get("mapping_letters", {}).values() if g != "MONO"}),
            "n_groups_optimal": (len({g for g in _g("mapping_optimal").values() if g != "MONO"})
                                 if _g("mapping_optimal") else None),
        }
        json.dump(_jsonable(cfg), open(SAVE_DIR / "config.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    _save("config.json (settings)", _cfg)

    # ── 2. Frame clusters + UMAP ──
    _save("results.pkl (clusters, superclusters, UMAP, X, per window)",
          lambda: pickle.dump(_g("results"), open(SAVE_DIR / "results.pkl", "wb")) if _g("results") else False)
    _save("df.pkl (current df)",
          lambda: _g("df").to_pickle(SAVE_DIR / "df.pkl") if _g("df") is not None else False)

    # ── 3. Sequences + SeqVec distance ──
    def _seqs():
        fs = _g("final_state")
        if not fs:
            return False
        pickle.dump(fs, open(SAVE_DIR / "final_state.pkl", "wb"))
        json.dump({a: {"seq": [int(c) for c in v["seq"]], "lengths": [int(x) for x in v["lengths"]],
                       "total": int(v.get("total", sum(v["lengths"])))} for a, v in fs["seqs"].items()},
                  open(SAVE_DIR / "sequences.json", "w"), indent=1)
        np.save(SAVE_DIR / "D_seqvec.npy", np.asarray(fs["D"]))
        json.dump([str(a) for a in fs["multi"]], open(SAVE_DIR / "D_seqvec_ids.json", "w"))
    _save("final_state.pkl + sequences.json + D_seqvec.npy (sequences, distances, SeqVec weights)", _seqs)

    # ── 4. Groups ──
    _save("mapping_letters.json (explorer groups)",
          lambda: json.dump(_jsonable(_g("final_state")["mapping_letters"]),
                            open(SAVE_DIR / "mapping_letters.json", "w"), indent=1) if _g("final_state") else False)
    def _opt():
        if not _g("mapping_optimal"):
            return False
        json.dump(_jsonable(_g("mapping_optimal")), open(SAVE_DIR / "mapping_optimal.json", "w"), indent=1)
        json.dump(_jsonable(_g("mapping_optimal")), open(OUT_DIR / "mapping_optimal_cuttle.json", "w"), indent=2)
    _save("mapping_optimal.json (regrouped groups)", _opt)
    for _nm in ("final_state_optimal", "history", "profiles"):
        _save(f"{_nm}.pkl", lambda _nm=_nm: pickle.dump(_g(_nm), open(SAVE_DIR / f"{_nm}.pkl", "wb"))
              if _g(_nm) is not None else False)
    for _nm, _fn in (("df_pairs", "df_pairs_power.csv"), ("df_merged", "df_merged_comparative.csv")):
        _save(_fn, lambda _nm=_nm, _fn=_fn: _g(_nm).to_csv(SAVE_DIR / _fn, index=False)
              if _g(_nm) is not None else False)

    # ── 5. Group explanation tables (+ figures) ──
    _expl_out = io.StringIO()
    def _figsaver(prefix):
        def _show(*a, **k):
            for num in plt.get_fignums():
                f = plt.figure(num)
                f.savefig(FIG_DIR / f"{prefix}_{num}.png", dpi=110, bbox_inches="tight")
                plt.close(f)
        return _show

    def _explain():
        if "explain_groups" not in globals() or not _g("final_state"):
            return False
        _show0 = plt.show
        try:
            plt.show = _figsaver("explication_groupes")
            with contextlib.redirect_stdout(_expl_out):
                ex = explain_groups(figure=True)
        finally:
            plt.show = _show0
        pickle.dump(ex, open(SAVE_DIR / "explication_groupes.pkl", "wb"))
        (SAVE_DIR / "explication_groupes.txt").write_text(_expl_out.getvalue(), encoding="utf-8")
        ex["medians"].to_csv(SAVE_DIR / "groupes_medianes.csv")
        ex["zscores"].to_csv(SAVE_DIR / "groupes_scores_z.csv")
        ex["families"].to_csv(SAVE_DIR / "groupes_familles.csv")
        tab = ex["individuals"].copy()
        tab["explorer group"] = [_g("final_state")["mapping_letters"].get(a, "") for a in tab.index]
        if _g("mapping_optimal"):
            tab["regrouped group"] = [_g("mapping_optimal").get(a, "") for a in tab.index]
        tab.to_csv(SAVE_DIR / "groupes_individus.csv")
    _save("group explanation (tables + figures)", _explain)

    # ── 6. Figures ──
    def _umap_fig():
        res, w = _g("results"), _g("SELECTED_WINDOW")
        if not res:
            return False
        w = w if w in res else list(res)[0]
        d, Zu = res[w]["df"], res[w]["Z"]
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for ax, col in zip(axes, ("cluster", "supercluster")):
            if col in d:
                labs = d[col].to_numpy()
                for k in np.unique(labs):
                    m = labs == k
                    ax.scatter(Zu[m, 0], Zu[m, 1], s=1, alpha=.4, color=plt.cm.tab20(int(k) % 20), label=str(k))
                ax.legend(markerscale=8, fontsize=6, ncol=2, title=col)
                ax.set_title(f"UMAP — {col} (window {w})")
        fig.savefig(FIG_DIR / "umap_clusters.png", dpi=110, bbox_inches="tight")
        plt.close(fig)
    _save("figures/umap_clusters.png", _umap_fig)

    def _mds_fig():
        if "visualize_strategy_groups" not in globals() or not _g("final_state"):
            return False
        _show0 = plt.show
        try:
            plt.show = _figsaver("groupes_mds")
            with contextlib.redirect_stdout(io.StringIO()):
                visualize_strategy_groups(_g("final_state"), method="mds")
        finally:
            plt.show = _show0
    _save("figures/groupes_mds_*.png", _mds_fig)

    # ── 7. Copy of the code (notebook without outputs + add_features_full.py) ──
    def _code():
        copied = []
        for nm in ("all_cuttlefish_english.ipynb", "add_features_full.py"):
            src = Path.cwd() / nm
            if not src.exists():
                continue
            if nm.endswith(".ipynb"):
                nb = json.load(open(src, encoding="utf-8"))
                for c in nb["cells"]:
                    if c["cell_type"] == "code":
                        c["outputs"], c["execution_count"] = [], None
                json.dump(nb, open(SAVE_DIR / f"code_{nm}", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
            else:
                shutil.copy(src, SAVE_DIR / f"code_{nm}")
            copied.append(nm)
        return copied or False
    _save("code (notebook without outputs + add_features)", _code)

    # ── Summary ──
    if _was_interactive:
        plt.ion()
    files = sorted(p for p in SAVE_DIR.rglob("*") if p.is_file())
    print(f"\n{'=' * 60}\n  {SAVE_NAME} : {len(files)} files ({sum(p.stat().st_size for p in files) / 1e6:.0f} Mo)")
    for p in files:
        print(f"    {str(p.relative_to(SAVE_DIR)):<45} {p.stat().st_size / 1024:>9.0f} Ko")
    bad = [l for s, l in _report if s == "ERR"]
    if bad:
        print(f"\n  ⚠️  items in error : {bad}")
    print("=" * 60)

[SAVE] disabled (DO_SAVE = False): nothing was written. To save: DO_SAVE = True, then rerun this cell.


In [19]:
# ══════════════════════════════════════════════════════════════
#  RELOADING A SAVED SESSION
#  Run after cell 0 (and the cells defining the functions you need).
#  Do NOT rerun cell 2 afterwards: it would replace the reloaded clusters with the current cache.
# ══════════════════════════════════════════════════════════════
DO_LOAD        = False         # ⚠ SAFETY: set True then rerun THIS cell to reload
LOAD_NAME      = None            # folder name to reload; None = the most recent session
RESTORE_CONFIG = True            # True = also restores FEATURES, weights, Leiden, exclusions, orbit threshold of the session

if not DO_LOAD:
    print("[LOAD] disabled (DO_LOAD = False): memory was not changed. "
          "To reload: DO_LOAD = True, then rerun this cell.")
else:

    import json, pickle
    from pathlib import Path
    import numpy as np
    import pandas as pd

    SESSION_ROOT = RESULTS_DIR
    sessions = sorted([d for d in SESSION_ROOT.iterdir() if d.is_dir() and (d / "config.json").exists()], key=lambda d: d.stat().st_mtime)
    if not sessions:
        raise FileNotFoundError(f"No session in {SESSION_ROOT}")
    print("Available sessions :\n")
    for s in sessions:
        c = json.load(open(s / "config.json", encoding="utf-8")) if (s / "config.json").exists() else {}
        print(f"  • {s.name:<28} {c.get('date', '')}  groups explorer={c.get('n_groups_explorer', '?')}"
              f" / optimal={c.get('n_groups_optimal', '?')}  {c.get('note', '')}")
    LOAD_DIR = SESSION_ROOT / LOAD_NAME if LOAD_NAME else sessions[-1]
    if not LOAD_DIR.exists():
        raise FileNotFoundError(f"Session not found : {LOAD_DIR}")
    print(f"\n[LOAD] → {LOAD_DIR.name}\n")
    loaded, skipped = [], []


    def _load(label, fn):
        try:
            out = fn()
            if out is False:
                skipped.append(label); print(f"  [SKIP] {label}")
            else:
                loaded.append(label); print(f"  [OK]   {label}")
        except FileNotFoundError:
            skipped.append(label); print(f"  [SKIP] {label}  (file missing)")
        except Exception as e:
            skipped.append(label); print(f"  [ERR]  {label}  ({type(e).__name__}: {e})")


    # ── Configuration ──
    cfg = json.load(open(LOAD_DIR / "config.json", encoding="utf-8")) if (LOAD_DIR / "config.json").exists() else {}
    if cfg.get("DATASET") and cfg["DATASET"] != globals().get("DATASET"):
        print(f"  ⚠️  different DATASET : session={cfg['DATASET']} | current={globals().get('DATASET')}")
    def _load_cfg():
        if not RESTORE_CONFIG or not cfg:
            return False
        for k in ("SUBJECT", "SPEED_COL", "SELECTED_WINDOW", "FEATURES", "FEATURE_WEIGHTS", "EXCLUDE_ANIMALS", "ORBIT_MIN_DEG_S",
                  "LEIDEN_K", "LEIDEN_RES", "LEIDEN_K2", "LEIDEN_RES2", "FRAME_STEP"):
            if cfg.get(k) is not None:
                v = cfg[k]
                if k == "SELECTED_WINDOW":
                    v = int(v)
                globals()[k] = v
    _load("config.json (features, weights, Leiden, exclusions, orbit threshold)", _load_cfg)
    # videos matching the frame numbering of the session
    #   shrimp: full videos (01_VIDEO_Cropped) | cuttlefish cut: videos_catch (videoN_cut.mp4)
    if globals().get("SUBJECT") == "shrimp" or cfg.get("DATASET") in ("shrimp", "full"):
        VIDEO_DIR, USE_FULL = VIDEO_DIR_FULL, True
    elif cfg.get("DATASET") in ("cut_new", "cut_old"):
        VIDEO_DIR, USE_FULL = VIDEO_DIR_CUT, False
    print(f"  [OK]   videos of this session → {VIDEO_DIR}")

    # ── Frame clusters + UMAP ──
    def _load_results():
        global results, X, Z, n_cl, n_su, df
        if (LOAD_DIR / "results.pkl").exists():
            results = pickle.load(open(LOAD_DIR / "results.pkl", "rb"))
        else:                                            # old save format
            meta = json.load(open(LOAD_DIR / "results_meta.json"))
            results = {int(w): {"df": pd.read_csv(LOAD_DIR / f"df_window_{w}.csv"),
                                "X": np.load(LOAD_DIR / f"X_window_{w}.npy"),
                                "Z": np.load(LOAD_DIR / f"Z_umap_{w}.npy"),
                                "n_clusters": m["n_cl"], "n_superclusters": m["n_su"]} for w, m in meta.items()}
        w = globals().get("SELECTED_WINDOW", list(results)[0])
        w = w if w in results else list(results)[0]
        X, Z = results[w]["X"], results[w]["Z"]
        n_cl, n_su = results[w]["n_clusters"], results[w]["n_superclusters"]
        df = results[w]["df"]
    _load("results (clusters, superclusters, UMAP)", _load_results)
    def _load_df():
        global df
        if (LOAD_DIR / "df.pkl").exists():
            df = pd.read_pickle(LOAD_DIR / "df.pkl")
        elif (LOAD_DIR / "df_main.csv").exists():
            df = pd.read_csv(LOAD_DIR / "df_main.csv")
        else:
            return False
    _load("df (current df)", _load_df)

    # ── Sequences, distances and groups ──
    def _load_pickle(name, required=False):
        def _f():
            p = LOAD_DIR / f"{name}.pkl"
            if not p.exists():
                globals().pop(name, None)            # missing from the session -> removed from memory (no mixing of sessions)
                return False
            globals()[name] = pickle.load(open(p, "rb"))
        return _f
    _load("final_state (sequences, SeqVec distance, explorer groups)", _load_pickle("final_state"))
    def _load_mapping():
        p = LOAD_DIR / "mapping_optimal.json"
        if not p.exists():
            globals().pop("mapping_optimal", None)   # missing from the session -> removed from memory (no mixing of sessions)
            return False
        globals()["mapping_optimal"] = json.load(open(p))
    _load("mapping_optimal (regrouped groups)", _load_mapping)
    for _nm in ("final_state_optimal", "history", "profiles", "explication_groupes"):
        _load(_nm, _load_pickle(_nm))
    if "explication_groupes" in globals():
        explication = explication_groupes
    def _load_d():
        global D_seqvec, D_seqvec_ids
        D_seqvec = np.load(LOAD_DIR / "D_seqvec.npy")
        D_seqvec_ids = json.load(open(LOAD_DIR / "D_seqvec_ids.json")) if (LOAD_DIR / "D_seqvec_ids.json").exists() else None
    _load("D_seqvec", _load_d)
    for _nm, _fn in (("df_pairs", "df_pairs_power.csv"), ("df_merged", "df_merged_comparative.csv")):
        _load(_nm, lambda _nm=_nm, _fn=_fn: globals().__setitem__(_nm, pd.read_csv(LOAD_DIR / _fn)))
    if "final_state" in globals():
        mapping_letters = final_state.get("mapping_letters", {})

    # ── Summary ──
    print(f"\n{'=' * 60}\nSESSION RELOADED — {LOAD_DIR.name}  ({len(loaded)} loaded | {len(skipped)} missing)")
    checks = {
        "results / df":     lambda: f"{len(results)} window(s) | df {df.shape}",
        "final_state":      lambda: f"{final_state['n_cl']} states | {final_state['n_s']} groups | {final_state.get('desc', '')}",
        "mapping_optimal":  lambda: f"groups = {sorted(set(mapping_optimal.values()))}",
        "SeqVec":           lambda: f"{final_state.get('seqvec_weights')} | knn_k={final_state.get('knn_k')}",
        "FEATURE_WEIGHTS":  lambda: f"{FEATURE_WEIGHTS}",
        "LEIDEN":           lambda: f"K={LEIDEN_K} res={LEIDEN_RES} | orbit_min={globals().get('ORBIT_MIN_DEG_S')}",
    }
    for nm, fn in checks.items():
        try:
            print(f"    ✅  {nm:<18} {fn()}")
        except Exception:
            print(f"    —   {nm:<18} not defined")
    print(f"\n  Then: run the analysis / video / explain_groups() cells directly.\n  To recompute from these settings: rerun cell 2 then the explorer.\n{'=' * 60}")

[LOAD] disabled (DO_LOAD = False): memory was not changed. To reload: DO_LOAD = True, then rerun this cell.
